In [ ]:
# SINGLE CONFIGURATION BLOCK — edit only this cell before a run.
RUN_MODE = "full"  # smoke or full
OUTPUT_DIR = "/content/counterfactual_faithfulness_stage9"
SEED = 913
MODEL_NAME = ["jepa_wm_pusht", "jepa_wm_wall"]
ENVIRONMENT = ["PushT", "Wall"]
HORIZONS = [1, 3, 6]
NUM_STATES = 24  # per environment in smoke mode
ACTIONS_PER_STATE = 10

MOUNT_DRIVE = False
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/counterfactual_faithfulness_stage9"
REUSE_STAGE7_CACHE = True
STAGE7_OUTPUT_DIR = "/content/counterfactual_faithfulness_stage7"
STAGE7_DRIVE_OUTPUT_DIR = (
    "/content/drive/MyDrive/counterfactual_faithfulness_stage7"
)
REPO_URL = "https://github.com/facebookresearch/jepa-wms.git"
REPO_COMMIT = "13cf1d9c7e476f53c17714d2e0f1dc239a883ce0"
FRAMESKIP = 5
TARGET_STEPS = list(range(1, max(HORIZONS) + 1))
TASKS_PER_ENVIRONMENT = 12
TASK_SPLIT_COUNTS = [6, 3, 0, 3]
EVALUATION_SEEDS = [913, 1297, 1709]
RIDGE_LAMBDAS = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
BOOTSTRAP_REPS = 200
RANKING_TIE = 1e-9

# Kept compatible with the Stage 7 transition cache.
AUDIT_PROJECTION_DIM = 64
AUDIT_PROJECTION_SEEDS = [7101]
GOAL_PROJECTION_DIM = 32
GOAL_PROJECTION_SEEDS = [8101, 10101]
ENERGY_HEAD_SEEDS = [8301]
ENERGY_METHODS = []
ENERGY_HIDDEN_DIM = 96
ENERGY_DROPOUT = 0.10
ENERGY_IMPLEMENTATION_ID = "unused_stage9_compatibility"
TRAINING_BATCH_STATES = 1
TRAINING_LR = 3e-4
TRAINING_WEIGHT_DECAY = 1e-3
PAIRWISE_WEIGHT = 1.0
LISTWISE_WEIGHT = 1.0
COST_SHAPE_WEIGHT = 0.25
PAIRWISE_TEMPERATURE = 0.25
LISTWISE_TEMPERATURE = 0.20

# Stage 9: CAVE-JEPA action-path adaptation.
ADAPTATION_METHODS = [
    "latent_only_action_path",
    "shuffled_cave_action_path",
    "matched_cave_action_path",
]
ADAPTATION_SEED = 9401
ADAPTATION_EPOCHS = 2
SELECTION_EPOCHS = [1, 2]
TRAINING_EPOCHS = ADAPTATION_EPOCHS
ADAPTATION_LR = 1e-5
ADAPTATION_WEIGHT_DECAY = 1e-4
GRADIENT_CLIP = 1.0
READOUT_PROJECTION_DIM = 128
EVALUATION_PROJECTION_SEED = 11903
ACTION_GROUP_SIZE = 6
LOSS_WEIGHTS = {
    "anchor": 0.25,
    "state": 1.0,
    "effect": 2.0,
    "action": 0.25,
    "proximity": 1e-4,
}
USE_ACTIVATION_CHECKPOINTING = True
NATIVE_NONINFERIORITY_TOLERANCE = 0.02

DOWNLOAD_RESULTS = True
EVIDENCE_STATUS = "EXPLORATORY_METHOD_DEVELOPMENT"
TASK_FAMILY_ID = "stage5_tasks_reused_for_stage9_development"
DEVELOPMENT_SPLIT = "development_holdout"

if RUN_MODE == "full":
    NUM_STATES = 96
    AUDIT_PROJECTION_DIM = 128
    AUDIT_PROJECTION_SEEDS = [7101, 9101]
    READOUT_PROJECTION_DIM = 256
    ADAPTATION_EPOCHS = 12
    TRAINING_EPOCHS = ADAPTATION_EPOCHS
    SELECTION_EPOCHS = [4, 8, 12]
    BOOTSTRAP_REPS = 2000
elif RUN_MODE != "smoke":
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")

assert MODEL_NAME == ["jepa_wm_pusht", "jepa_wm_wall"]
assert ENVIRONMENT == ["PushT", "Wall"]
assert HORIZONS == [1, 3, 6]
assert TARGET_STEPS == [1, 2, 3, 4, 5, 6]
assert ACTIONS_PER_STATE == 10
assert NUM_STATES % TASKS_PER_ENVIRONMENT == 0
assert sum(TASK_SPLIT_COUNTS) == TASKS_PER_ENVIRONMENT
assert SELECTION_EPOCHS[-1] == ADAPTATION_EPOCHS
assert 2 <= ACTION_GROUP_SIZE <= ACTIONS_PER_STATE


# Stage 9: counterfactual value-equivalent AdaLN adaptation

This notebook tests a different class of fix from Stages 5--8.

Those stages kept the JEPA transition function frozen, or corrected its output
after the transition was already formed. Stage 9 updates only the real
action-conditioning pathway in the pinned JEPA-WM predictor: the action
encoder and the six AdaLN modulation maps. The visual encoder and all
state/content weights remain frozen.

For each exactly restored simulator state, the notebook rolls out a null branch
and nine alternative action sequences. It trains predicted latent differences
to represent the corresponding **goal-independent physical outcome
differences**. The native latent target remains as an anchor.

The core loss is

\[
\mathcal L =
\lambda_z L_{\rm latent}
+\lambda_x L_{\rm state}
+\lambda_\Delta L_{\rm same\text{-}state\ effect}
+\lambda_a L_{\rm displacement\ action}.
\]

The temporary semantic and action heads are discarded. Every predictor is then
evaluated through a new, identical linear physical-state readout fitted only on
the probe-training tasks. This makes the test about transition geometry rather
than head memorization.

Controls:

- ordinary latent-only adaptation of the identical parameters;
- a deterministic within-state shuffle of non-null action/outcome assignment;
- the frozen predictor;
- native goal-latent distance and simulator oracle.

This is exploratory method development on an inspected task family. A positive
result must be followed by a new-task, multi-seed confirmation.


In [ ]:
import subprocess
import sys

# Keep Colab's CUDA-matched torch and torchvision builds.
PINNED = [
    "einops==0.8.1",
    "tensordict==0.9.1",
    "timm==1.0.19",
    "omegaconf==2.3.0",
    "hydra-core==1.3.2",
    "PyYAML==6.0.2",
    "huggingface_hub==0.36.2",
    "hf-xet==1.5.1",
    "gym==0.23.1",
    "pygame==2.6.1",
    "pymunk==6.8.0",
    "opencv-python-headless==4.11.0.86",
    "shapely==2.1.2",
    "lpips==0.1.4",
    "ruamel.yaml==0.18.10",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", *PINNED],
    check=True,
)
print("Installed pinned non-PyTorch dependencies.")
print("No runtime restart is expected.")


In [ ]:
import csv
import gc
import hashlib
import json
import logging
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import traceback
import zipfile
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as torch_functional
import torchvision
import yaml

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = DRIVE_OUTPUT_DIR
    STAGE7_OUTPUT_DIR = STAGE7_DRIVE_OUTPUT_DIR

OUT = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)
LOCAL_INTERMEDIATE = OUT / "intermediate"
STAGE7_ROOT = Path(STAGE7_OUTPUT_DIR)


def stage7_cache_compatible():
    if not REUSE_STAGE7_CACHE:
        return False, "disabled"
    config_path = STAGE7_ROOT / "config.json"
    intermediate = STAGE7_ROOT / "intermediate"
    if not config_path.exists() or not intermediate.exists():
        return False, "stage7 cache not found"
    previous = json.loads(config_path.read_text())
    required = {
        "SEED": SEED,
        "MODEL_NAME": MODEL_NAME,
        "ENVIRONMENT": ENVIRONMENT,
        "HORIZONS": HORIZONS,
        "NUM_STATES": NUM_STATES,
        "ACTIONS_PER_STATE": ACTIONS_PER_STATE,
        "REPO_COMMIT": REPO_COMMIT,
        "FRAMESKIP": FRAMESKIP,
        "TARGET_STEPS": TARGET_STEPS,
        "TASKS_PER_ENVIRONMENT": TASKS_PER_ENVIRONMENT,
        "TASK_SPLIT_COUNTS": TASK_SPLIT_COUNTS,
        "AUDIT_PROJECTION_DIM": AUDIT_PROJECTION_DIM,
        "AUDIT_PROJECTION_SEEDS": AUDIT_PROJECTION_SEEDS,
    }
    mismatch = [
        key for key, value in required.items()
        if previous.get(key) != value
    ]
    if mismatch:
        return False, f"incompatible keys: {mismatch}"
    expected = [
        intermediate / "truth" / environment.lower() / f"state_{state_id:04d}.npz"
        for environment in ENVIRONMENT
        for state_id in range(NUM_STATES)
    ]
    expected.extend(
        intermediate / "transitions" / model_name / f"state_{state_id:04d}.npz"
        for model_name in MODEL_NAME
        for state_id in range(NUM_STATES)
    )
    expected.extend(
        intermediate / "goals" / f"{model_name}.npz"
        for model_name in MODEL_NAME
    )
    missing = [str(path) for path in expected if not path.exists()]
    if missing:
        return False, f"missing {len(missing)} required shards"
    return True, "compatible complete Stage 7 cache"


if (
    LOCAL_INTERMEDIATE.exists()
    and any(LOCAL_INTERMEDIATE.rglob("state_*.npz"))
):
    INTERMEDIATE = LOCAL_INTERMEDIATE
    CACHE_REUSED = False
    CACHE_REUSE_REASON = "resuming Stage 9 cache"
else:
    CACHE_REUSED, CACHE_REUSE_REASON = stage7_cache_compatible()
    INTERMEDIATE = (
        STAGE7_ROOT / "intermediate"
        if CACHE_REUSED
        else LOCAL_INTERMEDIATE
    )

TRUTH_ROOT = INTERMEDIATE / "truth"
MODEL_ROOT = INTERMEDIATE / "models"
LOG_DIR = OUT / "logs"
PLOT_DIR = OUT / "plots"
for path in [OUT, INTERMEDIATE, TRUTH_ROOT, MODEL_ROOT, LOG_DIR, PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = (
    Path("/content/drive/MyDrive/cf_faithfulness_cache")
    if MOUNT_DRIVE
    else Path("/content/cf_faithfulness_cache")
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_HOME"] = str(CACHE_ROOT / "torch")
os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["MPLCONFIGDIR"] = str(CACHE_ROOT / "matplotlib")
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["JEPAWM_LOGS"] = str(CACHE_ROOT / "unused_jepawm_logs")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass
if not torch.cuda.is_available():
    raise RuntimeError("A Colab GPU runtime is required.")
if tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) < (2, 7):
    raise RuntimeError(f"JEPA-WMs requires torch>=2.7; found {torch.__version__}")

VERSIONS = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "torchvision": torchvision.__version__,
    "cuda_runtime": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_total_bytes": torch.cuda.get_device_properties(0).total_memory,
}
print(json.dumps(VERSIONS, indent=2))
print(
    json.dumps(
        {
            "cache_reused": CACHE_REUSED,
            "intermediate": str(INTERMEDIATE),
            "reason": CACHE_REUSE_REASON,
        },
        indent=2,
    )
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "run.log"),
        logging.StreamHandler(sys.stdout),
    ],
    force=True,
)
log = logging.getLogger("stage9")
log.info(
    "Seeds set to %d; evaluation seeds=%s; energy seeds=%s",
    SEED,
    EVALUATION_SEEDS,
    ENERGY_HEAD_SEEDS,
)


def gpu_report(label):
    payload = {
        "label": label,
        "allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 3),
        "reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 3),
        "peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 3),
    }
    log.info("GPU memory %s", payload)
    return payload


CONFIG = {
    "RUN_MODE": RUN_MODE,
    "OUTPUT_DIR": OUTPUT_DIR,
    "SEED": SEED,
    "MODEL_NAME": MODEL_NAME,
    "ENVIRONMENT": ENVIRONMENT,
    "HORIZONS": HORIZONS,
    "NUM_STATES": NUM_STATES,
    "ACTIONS_PER_STATE": ACTIONS_PER_STATE,
    "MOUNT_DRIVE": MOUNT_DRIVE,
    "REUSE_STAGE7_CACHE": REUSE_STAGE7_CACHE,
    "STAGE7_OUTPUT_DIR": STAGE7_OUTPUT_DIR,
    "REPO_URL": REPO_URL,
    "REPO_COMMIT": REPO_COMMIT,
    "FRAMESKIP": FRAMESKIP,
    "TARGET_STEPS": TARGET_STEPS,
    "TASKS_PER_ENVIRONMENT": TASKS_PER_ENVIRONMENT,
    "TASK_SPLIT_COUNTS": TASK_SPLIT_COUNTS,
    "EVALUATION_SEEDS": EVALUATION_SEEDS,
    "RIDGE_LAMBDAS": RIDGE_LAMBDAS,
    "BOOTSTRAP_REPS": BOOTSTRAP_REPS,
    "RANKING_TIE": RANKING_TIE,
    "AUDIT_PROJECTION_DIM": AUDIT_PROJECTION_DIM,
    "AUDIT_PROJECTION_SEEDS": AUDIT_PROJECTION_SEEDS,
    "GOAL_PROJECTION_DIM": GOAL_PROJECTION_DIM,
    "GOAL_PROJECTION_SEEDS": GOAL_PROJECTION_SEEDS,
    "ENERGY_HEAD_SEEDS": ENERGY_HEAD_SEEDS,
    "ENERGY_METHODS": ENERGY_METHODS,
    "ENERGY_HIDDEN_DIM": ENERGY_HIDDEN_DIM,
    "ENERGY_DROPOUT": ENERGY_DROPOUT,
    "ENERGY_IMPLEMENTATION_ID": ENERGY_IMPLEMENTATION_ID,
    "TRAINING_EPOCHS": TRAINING_EPOCHS,
    "SELECTION_EPOCHS": SELECTION_EPOCHS,
    "TRAINING_BATCH_STATES": TRAINING_BATCH_STATES,
    "TRAINING_LR": TRAINING_LR,
    "TRAINING_WEIGHT_DECAY": TRAINING_WEIGHT_DECAY,
    "PAIRWISE_WEIGHT": PAIRWISE_WEIGHT,
    "LISTWISE_WEIGHT": LISTWISE_WEIGHT,
    "COST_SHAPE_WEIGHT": COST_SHAPE_WEIGHT,
    "PAIRWISE_TEMPERATURE": PAIRWISE_TEMPERATURE,
    "LISTWISE_TEMPERATURE": LISTWISE_TEMPERATURE,
    "DOWNLOAD_RESULTS": DOWNLOAD_RESULTS,
    "EVIDENCE_STATUS": EVIDENCE_STATUS,
    "TASK_FAMILY_ID": TASK_FAMILY_ID,
    "DEVELOPMENT_SPLIT": DEVELOPMENT_SPLIT,
    "ADAPTATION_METHODS": ADAPTATION_METHODS,
    "ADAPTATION_SEED": ADAPTATION_SEED,
    "ADAPTATION_EPOCHS": ADAPTATION_EPOCHS,
    "ADAPTATION_LR": ADAPTATION_LR,
    "ADAPTATION_WEIGHT_DECAY": ADAPTATION_WEIGHT_DECAY,
    "GRADIENT_CLIP": GRADIENT_CLIP,
    "READOUT_PROJECTION_DIM": READOUT_PROJECTION_DIM,
    "EVALUATION_PROJECTION_SEED": EVALUATION_PROJECTION_SEED,
    "ACTION_GROUP_SIZE": ACTION_GROUP_SIZE,
    "LOSS_WEIGHTS": LOSS_WEIGHTS,
    "USE_ACTIVATION_CHECKPOINTING": USE_ACTIVATION_CHECKPOINTING,
    "NATIVE_NONINFERIORITY_TOLERANCE": NATIVE_NONINFERIORITY_TOLERANCE,
    "pinned_dependencies": PINNED,
}
RUN_SIGNATURE = hashlib.sha256(
    json.dumps(CONFIG, sort_keys=True).encode()
).hexdigest()
CONFIG["run_signature"] = RUN_SIGNATURE
config_path = OUT / "config.json"
if config_path.exists():
    previous = json.loads(config_path.read_text())
    if previous.get("run_signature") != RUN_SIGNATURE:
        raise RuntimeError(
            "OUTPUT_DIR contains a different configuration; choose a new OUTPUT_DIR."
        )
config_path.write_text(json.dumps(CONFIG, indent=2) + "\n")
(OUT / "versions.json").write_text(json.dumps(VERSIONS, indent=2) + "\n")
(OUT / "cache_reuse.json").write_text(
    json.dumps(
        {
            "cache_reused": CACHE_REUSED,
            "source": str(INTERMEDIATE),
            "reason": CACHE_REUSE_REASON,
        },
        indent=2,
    )
    + "\n"
)
(OUT / "FAILURE_TRACE.txt").write_text("PENDING\n")

PIPELINE_FAILED = False
FAILURE_MESSAGE = ""


def record_failure(stage):
    global PIPELINE_FAILED, FAILURE_MESSAGE
    PIPELINE_FAILED = True
    FAILURE_MESSAGE = f"STAGE: {stage}\n{traceback.format_exc()}"
    (OUT / "FAILURE_TRACE.txt").write_text(FAILURE_MESSAGE)
    log.exception("Captured failure in %s", stage)


gpu_report("startup")

In [ ]:
MODEL_BY_ENVIRONMENT = {
    "PushT": ["jepa_wm_pusht"],
    "Wall": ["jepa_wm_wall"],
}
READOUTS = [
    "latent_distance",
    "linear_pose",
    "action_blind",
    "linear_pose_shuffled",
    "oracle_pose",
]
SPLIT_NAMES = [
    "probe_train",
    "probe_calibration",
    "regression_train",
    DEVELOPMENT_SPLIT,
]


def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, allow_nan=True) + "\n")


def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        if not rows:
            raise ValueError(f"cannot infer fields for empty table {path}")
        fieldnames = list(rows[0])
    temporary = Path(path).with_suffix(".tmp.csv")
    with temporary.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def atomic_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_suffix(".tmp.npz")
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)


def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_bytes):
            digest.update(chunk)
    return digest.hexdigest()


def pair_indices(n_actions):
    return np.triu_indices(n_actions, k=1)


def to_model_observation(visual, proprio):
    visual_tensor = torch.from_numpy(np.asarray(visual))
    proprio_tensor = torch.from_numpy(np.asarray(proprio))
    if visual_tensor.ndim == 3:
        visual_tensor = visual_tensor[None, None]
        proprio_tensor = proprio_tensor[None, None]
    elif visual_tensor.ndim != 5:
        raise ValueError(f"unexpected visual shape {tuple(visual_tensor.shape)}")
    visual_tensor = visual_tensor.permute(0, 1, 4, 2, 3)
    return {"visual": visual_tensor, "proprio": proprio_tensor}


def unit_vector(vector):
    vector = np.asarray(vector, dtype=np.float64)
    norm = np.linalg.norm(vector)
    if norm < 1e-12:
        raise ValueError("zero direction")
    return vector / norm


def rotate_vector(vector, degrees):
    radians = np.deg2rad(degrees)
    return np.array(
        [
            [np.cos(radians), -np.sin(radians)],
            [np.sin(radians), np.cos(radians)],
        ]
    ) @ np.asarray(vector)


def feature_metrics(truth, prediction, eps=1e-12):
    truth = np.asarray(truth, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    errors = prediction - truth
    ordinary = np.sqrt(np.mean(errors**2, axis=(0, 2)))
    common = np.mean(errors, axis=0)
    centered = errors - common[None, :, :]
    action_dependent = np.sqrt(np.mean(centered**2, axis=(0, 2)))
    left, right = pair_indices(truth.shape[0])
    truth_delta = truth[left] - truth[right]
    predicted_delta = prediction[left] - prediction[right]
    pair_error = predicted_delta - truth_delta
    pair_rmse = np.sqrt(np.mean(pair_error**2, axis=(0, 2)))
    pair_scale = np.sqrt(np.mean(truth_delta**2, axis=(0, 2)))
    normalized = pair_rmse / np.maximum(pair_scale, eps)
    dot = np.sum(truth_delta * predicted_delta, axis=-1)
    denominator = (
        np.linalg.norm(truth_delta, axis=-1)
        * np.linalg.norm(predicted_delta, axis=-1)
    )
    cosine = np.divide(
        dot,
        denominator,
        out=np.zeros_like(dot),
        where=denominator > eps,
    ).mean(axis=0)
    expected_pair_mse = (
        2 * truth.shape[0] / (truth.shape[0] - 1)
    ) * action_dependent**2
    return {
        "ordinary_feature_rmse": ordinary,
        "common_mode_feature_rmse": np.sqrt(np.mean(common**2, axis=-1)),
        "action_dependent_feature_rmse": action_dependent,
        "paired_feature_rmse": pair_rmse,
        "normalized_paired_feature_rmse": normalized,
        "paired_feature_cosine": cosine,
        "pair_identity_residual": pair_rmse**2 - expected_pair_mse,
    }


def ranking_metrics(true_cost, predicted_cost, tie=RANKING_TIE):
    truth = np.asarray(true_cost, dtype=np.float64)
    prediction = np.asarray(predicted_cost, dtype=np.float64)
    selected = int(np.argmin(prediction))
    oracle = int(np.argmin(truth))
    best = float(np.min(truth))
    chosen = float(truth[selected])
    spread = float(np.max(truth) - best)
    regret = chosen - best
    normalized_regret = regret / spread if spread > tie else 0.0
    left, right = pair_indices(len(truth))
    true_margin = truth[left] - truth[right]
    predicted_margin = prediction[left] - prediction[right]
    valid = np.abs(true_margin) > tie
    credit = np.full(len(left), np.nan)
    same = np.sign(true_margin) == np.sign(predicted_margin)
    credit[valid & same] = 1.0
    credit[valid & (np.abs(predicted_margin) <= tie)] = 0.5
    credit[valid & np.isnan(credit)] = 0.0
    weights = np.abs(true_margin)
    pairwise = float(np.nanmean(credit)) if np.any(valid) else float("nan")
    weighted = (
        float(np.nansum(weights * credit) / np.sum(weights[valid]))
        if np.any(valid)
        else float("nan")
    )
    margin_scale = (
        float(np.sqrt(np.mean(true_margin[valid] ** 2))) if np.any(valid) else 0.0
    )
    normalized_margin_rmse = (
        float(
            np.sqrt(
                np.mean((predicted_margin[valid] - true_margin[valid]) ** 2)
            )
            / margin_scale
        )
        if margin_scale > tie
        else float("nan")
    )
    return {
        "selected_action": selected,
        "oracle_action": oracle,
        "top1_correct": float(chosen <= best + tie),
        "regret": float(regret),
        "normalized_regret": float(normalized_regret),
        "pairwise_accuracy": pairwise,
        "weighted_pairwise_accuracy": weighted,
        "normalized_margin_rmse": normalized_margin_rmse,
        "pair_left": left,
        "pair_right": right,
        "true_margin": true_margin,
        "predicted_margin": predicted_margin,
        "pair_credit": credit,
        "pair_weight": weights,
    }


def bootstrap_mean(values, groups, repetitions, seed):
    values = np.asarray(values, dtype=np.float64)
    groups = np.asarray(groups)
    finite = np.isfinite(values)
    values = values[finite]
    groups = groups[finite]
    unique = np.unique(groups)
    if len(unique) == 0:
        return {
            "estimate": float("nan"),
            "low": float("nan"),
            "high": float("nan"),
            "n_clusters": 0,
            "n_bootstrap": repetitions,
        }
    grouped = [values[groups == group] for group in unique]
    rng = np.random.default_rng(seed)
    draws = np.empty(repetitions)
    for index in range(repetitions):
        sampled = rng.integers(0, len(unique), size=len(unique))
        draws[index] = np.mean(
            np.concatenate([grouped[item] for item in sampled])
        )
    return {
        "estimate": float(np.mean(values)),
        "low": float(np.quantile(draws, 0.025)),
        "high": float(np.quantile(draws, 0.975)),
        "n_clusters": int(len(unique)),
        "n_bootstrap": int(repetitions),
    }


def random_projection(input_dim, output_dim, seed):
    rng = np.random.default_rng(seed)
    projection = rng.standard_normal(
        (input_dim, output_dim)
    ).astype(np.float32)
    scale = np.float32(np.sqrt(output_dim))
    return (projection / scale).astype(np.float32, copy=False)


def standardize_fit(values):
    values = np.asarray(values, dtype=np.float64)
    mean = np.mean(values, axis=0)
    scale = np.std(values, axis=0)
    scale[scale < 1e-8] = 1.0
    return mean, scale


def fit_linear_readout(x_train, y_train, x_calibration, y_calibration):
    mean, scale = standardize_fit(x_train)
    train = (np.asarray(x_train, dtype=np.float64) - mean) / scale
    calibration = (np.asarray(x_calibration, dtype=np.float64) - mean) / scale
    train = np.column_stack([np.ones(len(train)), train])
    calibration = np.column_stack([np.ones(len(calibration)), calibration])
    gram = train.T @ train
    cross = train.T @ np.asarray(y_train, dtype=np.float64)
    best = None
    for ridge in RIDGE_LAMBDAS:
        penalty = np.eye(gram.shape[0]) * ridge
        penalty[0, 0] = 0.0
        coefficient = np.linalg.solve(gram + penalty, cross)
        prediction = calibration @ coefficient
        loss = float(np.mean((prediction - y_calibration) ** 2))
        candidate = {
            "ridge": float(ridge),
            "calibration_pose_mse": loss,
            "mean": mean,
            "scale": scale,
            "coefficient": coefficient,
        }
        if best is None or loss < best["calibration_pose_mse"]:
            best = candidate
    return best


def predict_linear_readout(probe, values):
    standardized = (
        np.asarray(values, dtype=np.float64) - probe["mean"]
    ) / probe["scale"]
    augmented = np.column_stack([np.ones(len(standardized)), standardized])
    return augmented @ probe["coefficient"]


def configure_repo():
    repo = CACHE_ROOT / "jepa-wms"
    if not repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    subprocess.run(
        ["git", "-C", str(repo), "fetch", "origin", REPO_COMMIT],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(repo), "checkout", "--detach", REPO_COMMIT],
        check=True,
    )
    resolved = subprocess.check_output(
        ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
    ).strip()
    if resolved != REPO_COMMIT:
        raise RuntimeError(f"repository pin mismatch: {resolved}")

    # The public hub loader imports a planning entry point with extra evaluation
    # dependencies. Use its equivalent lightweight model constructor.
    hubconf = repo / "hubconf.py"
    text = hubconf.read_text()
    old = "from evals.simu_env_planning.eval import init_module"
    new = (
        "from app.vjepa_wm.modelcustom.simu_env_planning."
        "vit_enc_preds import init_module"
    )
    if old in text:
        hubconf.write_text(text.replace(old, new))
    elif new not in text:
        raise RuntimeError("unexpected hubconf.py at pinned commit")

    # PushT and Wall do not use the DROID pose helper.
    model_file = (
        repo / "app/vjepa_wm/modelcustom/simu_env_planning/vit_enc_preds.py"
    )
    text = model_file.read_text()
    old = "from app.plan_common.datasets.droid_dset import compute_new_pose"
    new = (
        "def compute_new_pose(*args, **kwargs):\n"
        "    raise RuntimeError('compute_new_pose is unused in simulator predict_proprio mode')"
    )
    if old in text:
        model_file.write_text(text.replace(old, new))
    elif "compute_new_pose is unused" not in text:
        raise RuntimeError("unexpected vit_enc_preds.py at pinned commit")

    config_paths = [
        repo
        / "configs/evals/simu_env_planning/pt/dino-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/pt/jepa-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/wall/dino-wm/"
        "wall_L2_cem_sourcerandstate_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/wall/jepa-wm/"
        "wall_L2_cem_sourcerandstate_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
    ]
    for config_path in config_paths:
        config = yaml.safe_load(config_path.read_text())
        heads = config["model_kwargs"]["pretrain_kwargs"]["heads_cfg"]
        heads["architectures"] = {}
        heads["pretrain_dec_path"] = None
        config_path.write_text(yaml.safe_dump(config, sort_keys=False))
    return repo


def task_split_map():
    rng = np.random.default_rng(SEED + 17)
    order = rng.permutation(TASKS_PER_ENVIRONMENT).tolist()
    mapping = {}
    start = 0
    for name, count in zip(SPLIT_NAMES, TASK_SPLIT_COUNTS):
        for task_id in order[start : start + count]:
            mapping[int(task_id)] = name
        start += count
    return mapping


def pusht_tasks():
    # Frozen before any Stage 5 simulation. None appears in the Stage 3 family.
    values = [
        (220.0, 200.0, -0.90),
        (200.0, 272.0, 0.35),
        (224.0, 320.0, -0.30),
        (270.0, 198.0, 0.95),
        (292.0, 224.0, -0.45),
        (314.0, 270.0, 0.55),
        (292.0, 316.0, -1.00),
        (246.0, 310.0, 0.15),
        (230.0, 238.0, 1.25),
        (276.0, 244.0, -1.25),
        (314.0, 292.0, 0.85),
        (244.0, 278.0, -0.75),
    ]
    splits = task_split_map()
    return [
        {
            "environment": "PushT",
            "task_id": index,
            "task_name": f"stage5_pusht_goal_{index:02d}",
            "goal": list(value),
            "split": splits[index],
        }
        for index, value in enumerate(values)
    ]


def wall_tasks():
    # Frozen before any Stage 5 simulation. None appears in the Stage 3 family.
    layouts = [
        (26.0, 20.0, 54.0, 44.0),
        (28.0, 34.0, 54.0, 16.0),
        (30.0, 44.0, 54.0, 32.0),
        (33.0, 16.0, 10.0, 48.0),
        (35.0, 28.0, 10.0, 20.0),
        (38.0, 42.0, 10.0, 36.0),
        (25.0, 38.0, 55.0, 22.0),
        (31.0, 24.0, 55.0, 50.0),
        (36.0, 46.0, 11.0, 16.0),
        (39.0, 20.0, 11.0, 40.0),
        (29.0, 30.0, 54.0, 50.0),
        (34.0, 38.0, 10.0, 24.0),
    ]
    splits = task_split_map()
    return [
        {
            "environment": "Wall",
            "task_id": index,
            "task_name": f"stage5_wall_layout_goal_{index:02d}",
            "wall_x": wall_x,
            "door_y": door_y,
            "goal": [goal_x, goal_y],
            "split": splits[index],
        }
        for index, (wall_x, door_y, goal_x, goal_y) in enumerate(layouts)
    ]


TASKS = {"PushT": pusht_tasks(), "Wall": wall_tasks()}

# Fail before simulation if the declared split protocol and generated tasks
# ever drift apart again.
EXPECTED_TASK_SPLIT_COUNTS = {
    name: count
    for name, count in zip(SPLIT_NAMES, TASK_SPLIT_COUNTS)
    if count > 0
}
for environment, tasks in TASKS.items():
    observed = {
        name: sum(task["split"] == name for task in tasks)
        for name in {task["split"] for task in tasks}
    }
    if observed != EXPECTED_TASK_SPLIT_COUNTS:
        raise AssertionError(
            f"{environment} task split mismatch: "
            f"expected {EXPECTED_TASK_SPLIT_COUNTS}, got {observed}"
        )


def build_state_records(environment):
    tasks = TASKS[environment]
    per_task = NUM_STATES // TASKS_PER_ENVIRONMENT
    records = []
    state_id = 0
    for task in tasks:
        for within_task in range(per_task):
            evaluation_seed = EVALUATION_SEEDS[within_task % len(EVALUATION_SEEDS)]
            rng = np.random.default_rng(
                evaluation_seed * 100000
                + task["task_id"] * 1000
                + within_task
            )
            if environment == "PushT":
                goal_xy = np.asarray(task["goal"][:2], dtype=np.float64)
                for _ in range(100):
                    radial = rng.uniform(85.0, 120.0)
                    polar = rng.uniform(-np.pi, np.pi)
                    block = goal_xy + radial * np.array(
                        [np.cos(polar), np.sin(polar)]
                    )
                    direction = unit_vector(goal_xy - block)
                    agent_distance = rng.uniform(58.0, 80.0)
                    agent = block - agent_distance * direction
                    if (
                        np.all(block > 90.0)
                        and np.all(block < 422.0)
                        and np.all(agent > 35.0)
                        and np.all(agent < 477.0)
                    ):
                        break
                else:
                    raise RuntimeError("could not build bounded PushT state")
                state = np.array(
                    [
                        agent[0],
                        agent[1],
                        block[0],
                        block[1],
                        rng.uniform(-0.65, 0.65),
                        0.0,
                        0.0,
                    ],
                    dtype=np.float64,
                )
                stratum = "near" if agent_distance < 69.0 else "far"
            else:
                wall_x = float(task["wall_x"])
                goal_x = float(task["goal"][0])
                goal_on_right = goal_x > wall_x
                if goal_on_right:
                    x = rng.uniform(8.0, max(9.0, wall_x - 8.0))
                else:
                    x = rng.uniform(min(56.0, wall_x + 8.0), 57.0)
                y = rng.uniform(8.0, 57.0)
                state = np.array([x, y], dtype=np.float64)
                stratum = "left_to_right" if goal_on_right else "right_to_left"
            records.append(
                {
                    "environment": environment,
                    "state_id": state_id,
                    "task_id": task["task_id"],
                    "task_name": task["task_name"],
                    "split": task["split"],
                    "evaluation_seed": int(evaluation_seed),
                    "design_stratum": stratum,
                    "state": state,
                }
            )
            state_id += 1
    if len(records) != NUM_STATES:
        raise AssertionError("state-record count mismatch")
    return records


def pusht_candidate_library(state, task, primitive_steps):
    direction = unit_vector(
        np.asarray(task["goal"][:2]) - np.asarray(state)[2:4]
    )
    specifications = [("noop", 0.0, 0)]
    specifications.extend(
        (f"direct_{duration}", 0.0, duration)
        for duration in [8, 12, 16, 20, 24, 30]
    )
    for angle in [-20.0, 20.0]:
        for duration in [12, 18, 24]:
            specifications.append(
                (f"angle_{angle:+.0f}_{duration}", angle, duration)
            )
    for angle in [-40.0, 40.0]:
        for duration in [18, 24]:
            specifications.append(
                (f"angle_{angle:+.0f}_{duration}", angle, duration)
            )
    for angle in [-70.0, 70.0, 140.0, -140.0, 180.0]:
        specifications.append((f"angle_{angle:+.0f}_24", angle, 24))
    sequences = []
    for _, angle, duration in specifications:
        sequence = np.zeros((primitive_steps, 2), dtype=np.float32)
        if duration:
            sequence[:duration] = (
                0.14 * rotate_vector(direction, angle)
            ).astype(np.float32)
        sequences.append(sequence)
    selected = np.asarray(
        [0, 2, 4, 6, 8, 11, 14, 16, 17, 18],
        dtype=np.int64,
    )
    return (
        np.stack(sequences)[selected],
        [specifications[index][0] for index in selected],
        selected,
    )


def nominal_waypoint_sequence(state, waypoints, primitive_steps, magnitude=0.75):
    position = np.asarray(state, dtype=np.float64).copy()
    sequence = np.zeros((primitive_steps, 2), dtype=np.float32)
    waypoints = [np.asarray(point, dtype=np.float64) for point in waypoints]
    for step in range(primitive_steps):
        remaining = primitive_steps - step
        waypoint_index = min(
            len(waypoints) - 1,
            (step * len(waypoints)) // primitive_steps,
        )
        delta = waypoints[waypoint_index] - position
        if np.linalg.norm(delta) < 0.5 and waypoint_index + 1 < len(waypoints):
            waypoint_index += 1
            delta = waypoints[waypoint_index] - position
        if np.linalg.norm(delta) > 1e-8:
            action = magnitude * unit_vector(delta)
            sequence[step] = action.astype(np.float32)
            position = position + 2.0 * action
    return sequence


def wall_candidate_library(state, task, primitive_steps):
    state = np.asarray(state, dtype=np.float64)
    goal = np.asarray(task["goal"], dtype=np.float64)
    wall_x = float(task["wall_x"])
    door_y = float(task["door_y"])
    side = np.sign(goal[0] - state[0])
    door = np.array([wall_x + side * 1.0, door_y])
    directions = [
        ("noop", np.zeros((primitive_steps, 2), dtype=np.float32)),
        ("direct", nominal_waypoint_sequence(state, [goal], primitive_steps)),
        (
            "via_door",
            nominal_waypoint_sequence(state, [door, goal], primitive_steps),
        ),
        (
            "via_door_high",
            nominal_waypoint_sequence(
                state,
                [door + np.array([0.0, 3.0]), goal],
                primitive_steps,
            ),
        ),
        (
            "via_door_low",
            nominal_waypoint_sequence(
                state,
                [door - np.array([0.0, 3.0]), goal],
                primitive_steps,
            ),
        ),
    ]
    direct = unit_vector(goal - state)
    for angle in [-35.0, 35.0, -70.0, 70.0]:
        sequence = np.zeros((primitive_steps, 2), dtype=np.float32)
        sequence[:] = (0.75 * rotate_vector(direct, angle)).astype(np.float32)
        directions.append((f"angle_{angle:+.0f}", sequence))
    reverse = np.zeros((primitive_steps, 2), dtype=np.float32)
    reverse[:] = (-0.55 * direct).astype(np.float32)
    directions.append(("reverse", reverse))
    if len(directions) != ACTIONS_PER_STATE:
        raise AssertionError("Wall candidate count mismatch")
    return (
        np.stack([item[1] for item in directions]),
        [item[0] for item in directions],
        np.arange(ACTIONS_PER_STATE, dtype=np.int64),
    )


def candidate_library(environment, state, task, primitive_steps):
    if environment == "PushT":
        return pusht_candidate_library(state, task, primitive_steps)
    return wall_candidate_library(state, task, primitive_steps)


def task_cost(environment, states, task):
    states = np.asarray(states, dtype=np.float64)
    if environment == "PushT":
        goal = np.asarray(task["goal"], dtype=np.float64)
        angle_error = np.arctan2(
            np.sin(states[..., 4] - goal[2]),
            np.cos(states[..., 4] - goal[2]),
        )
        pieces = np.concatenate(
            [
                (states[..., 2:4] - goal[:2]) / 512.0,
                (angle_error / np.pi)[..., None],
            ],
            axis=-1,
        )
        return np.linalg.norm(pieces, axis=-1)
    goal = np.asarray(task["goal"], dtype=np.float64)
    return np.linalg.norm((states[..., :2] - goal) / 65.0, axis=-1)


def pose_target(environment, states):
    states = np.asarray(states, dtype=np.float64)
    if environment == "PushT":
        angle = states[..., 4]
        return np.stack(
            [
                states[..., 2] / 512.0,
                states[..., 3] / 512.0,
                np.sin(angle),
                np.cos(angle),
            ],
            axis=-1,
        )
    return states[..., :2] / 65.0


def decoded_task_cost(environment, prediction, task):
    prediction = np.asarray(prediction, dtype=np.float64)
    if environment == "PushT":
        goal = np.asarray(task["goal"], dtype=np.float64)
        angle = np.arctan2(prediction[..., 2], prediction[..., 3])
        angle_error = np.arctan2(
            np.sin(angle - goal[2]),
            np.cos(angle - goal[2]),
        )
        return np.linalg.norm(
            np.concatenate(
                [
                    prediction[..., :2] - goal[:2] / 512.0,
                    (angle_error / np.pi)[..., None],
                ],
                axis=-1,
            ),
            axis=-1,
        )
    return np.linalg.norm(
        prediction[..., :2] - np.asarray(task["goal"]) / 65.0,
        axis=-1,
    )


def physical_pose_error(environment, prediction, truth):
    prediction = np.asarray(prediction, dtype=np.float64)
    truth = np.asarray(truth, dtype=np.float64)
    if environment == "PushT":
        angle_prediction = np.arctan2(prediction[..., 2], prediction[..., 3])
        angle_truth = np.arctan2(truth[..., 2], truth[..., 3])
        angle_error = np.arctan2(
            np.sin(angle_prediction - angle_truth),
            np.cos(angle_prediction - angle_truth),
        )
        pieces = np.concatenate(
            [
                prediction[..., :2] - truth[..., :2],
                (angle_error / np.pi)[..., None],
            ],
            axis=-1,
        )
        return np.linalg.norm(pieces, axis=-1)
    return np.linalg.norm(prediction[..., :2] - truth[..., :2], axis=-1)


def make_environment(repo, environment, task=None):
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    if environment == "PushT":
        from evals.simu_env_planning.envs.pusht_env.pusht_env import PushTEnv

        return PushTEnv(
            with_velocity=True,
            with_target=True,
            render_size=224,
            relative=True,
            action_scale=100,
        )

    from evals.simu_env_planning.envs.wall_gym_wrap import DEFAULT_CFG
    from evals.simu_env_planning.envs.wall_env.envs.wall import DotWall

    if task is None:
        task = TASKS["Wall"][0]
    env = DotWall(
        rng=np.random.default_rng(SEED),
        wall_config=deepcopy(DEFAULT_CFG),
        fix_wall=True,
        cross_wall=False,
        device="cpu",
    )
    env.wall_x = torch.tensor(float(task["wall_x"]))
    env.hole_y = torch.tensor(float(task["door_y"]))
    env.left_wall_x = env.wall_x - env.wall_config.wall_width // 2
    env.right_wall_x = env.wall_x + env.wall_config.wall_width // 2
    return env


def wall_visual(env):
    value = env.render().float()[None]
    resized = torch_functional.interpolate(
        value,
        size=(224, 224),
        mode="bilinear",
        align_corners=False,
        antialias=True,
    )[0]
    return (
        torch.clamp(torch.round(resized), 0, 255)
        .to(torch.uint8)
        .permute(1, 2, 0)
        .cpu()
        .numpy()
    )


def reset_environment(repo, environment, task, state, seed):
    if environment == "PushT":
        env = make_environment(repo, environment)
        env.seed(seed)
        env.reset_to_state = np.asarray(state, dtype=np.float64).copy()
        observation, restored = env.reset()
        payload = {
            "visual": np.asarray(observation["visual"]).copy(),
            "proprio": np.asarray(observation["proprio"]).copy(),
        }
        return env, payload, np.asarray(restored).copy()

    env = make_environment(repo, environment, task)
    env.seed(seed)
    env.reset_to_state = torch.as_tensor(
        np.asarray(state, dtype=np.float32)
    )
    observation, restored = env.reset()
    payload = {
        "visual": wall_visual(env),
        "proprio": np.asarray(
            observation["proprio"].detach().cpu(), dtype=np.float32
        ),
    }
    return env, payload, np.asarray(restored.detach().cpu(), dtype=np.float32)


def rollout_branch(repo, environment, task, state, actions, seed):
    env, initial, restored = reset_environment(
        repo, environment, task, state, seed
    )
    wanted = set(TARGET_STEPS)
    observations = {}
    states = {}
    interactions = {}
    interaction_types = {}
    cumulative_interactions = 0
    cumulative_crossings = 0
    previous_state = np.asarray(restored, dtype=np.float64).copy()

    for step, action in enumerate(actions, start=1):
        if environment == "PushT":
            observation, _, _, info = env.step(action)
            current_state = np.asarray(info["state"]).copy()
            cumulative_interactions += int(info.get("n_contacts", 0))
            current_observation = {
                "visual": np.asarray(observation["visual"]).copy(),
                "proprio": np.asarray(observation["proprio"]).copy(),
            }
        else:
            observation, _, _, info = env.step(
                torch.as_tensor(action, dtype=torch.float32)
            )
            current_state = np.asarray(
                info["state"].detach().cpu(), dtype=np.float64
            )
            proposed = previous_state + 2.0 * np.asarray(action)
            if np.linalg.norm(current_state - proposed) > 1e-5:
                cumulative_interactions += 1
            wall_x = float(task["wall_x"])
            crossed = (
                (previous_state[0] - wall_x) * (current_state[0] - wall_x)
                < 0
            )
            cumulative_crossings += int(crossed)
            current_observation = {
                "visual": wall_visual(env),
                "proprio": np.asarray(
                    observation["proprio"].detach().cpu(),
                    dtype=np.float32,
                ),
            }
        previous_state = current_state.copy()
        if step % FRAMESKIP == 0:
            horizon = step // FRAMESKIP
            if horizon in wanted:
                observations[horizon] = current_observation
                states[horizon] = current_state
                interactions[horizon] = cumulative_interactions
                if environment == "PushT":
                    interaction_types[horizon] = (
                        "contact" if cumulative_interactions > 0 else "free"
                    )
                elif cumulative_interactions > 0:
                    interaction_types[horizon] = "collision"
                elif cumulative_crossings > 0:
                    interaction_types[horizon] = "door_cross"
                else:
                    interaction_types[horizon] = "free"
    if wanted != set(observations):
        raise RuntimeError(f"missing horizons: {wanted - set(observations)}")
    return (
        initial,
        restored,
        observations,
        states,
        interactions,
        interaction_types,
    )


def exact_restore_test(repo, environment, task, state, actions):
    endpoints = []
    images = []
    interactions = []
    for _ in range(3):
        initial, _, _, states, counts, kinds = rollout_branch(
            repo,
            environment,
            task,
            state,
            actions,
            SEED + 9000,
        )
        endpoints.append(states[max(HORIZONS)])
        images.append(initial["visual"])
        interactions.append(
            (counts[max(HORIZONS)], kinds[max(HORIZONS)])
        )
    result = {
        "environment": environment,
        "repeats": 3,
        "endpoint_bitwise_exact": all(
            np.array_equal(endpoints[0], item) for item in endpoints[1:]
        ),
        "initial_render_bitwise_exact": all(
            np.array_equal(images[0], item) for item in images[1:]
        ),
        "diagnostics_exact": all(
            interactions[0] == item for item in interactions[1:]
        ),
        "max_endpoint_abs_diff": float(
            max(
                np.max(np.abs(endpoints[0] - item))
                for item in endpoints[1:]
            )
        ),
    }
    if not all(
        result[key]
        for key in [
            "endpoint_bitwise_exact",
            "initial_render_bitwise_exact",
            "diagnostics_exact",
        ]
    ):
        raise AssertionError(f"exact restoration failed: {result}")
    return result


def goal_observation(repo, environment, task):
    if environment == "PushT":
        goal = task["goal"]
        state = np.array(
            [80.0, 450.0, goal[0], goal[1], goal[2], 0.0, 0.0]
        )
    else:
        state = np.asarray(task["goal"], dtype=np.float64)
    _, observation, _ = reset_environment(
        repo,
        environment,
        task,
        state,
        SEED + 11000 + task["task_id"],
    )
    return observation


In [ ]:
# Phase A — reconstruct or reuse the fixed development interventions.
def generate_simulator_truth():
    repo = configure_repo()
    task_payload = []
    split_payload = {
        "protocol": (
            "exploratory task-disjoint training/calibration/development; "
            "the previously inspected final partition is relabeled development_holdout"
        ),
        "environments": {},
    }
    restore_payload = {}
    design_payload = {}

    for environment in ENVIRONMENT:
        truth_dir = TRUTH_ROOT / environment.lower()
        truth_dir.mkdir(parents=True, exist_ok=True)
        records = build_state_records(environment)
        tasks_by_id = {
            item["task_id"]: item for item in TASKS[environment]
        }
        for task in TASKS[environment]:
            task_payload.append(task)

        split_payload["environments"][environment] = {
            name: {
                "task_ids": sorted(
                    task["task_id"]
                    for task in TASKS[environment]
                    if task["split"] == name
                ),
                "state_ids": sorted(
                    record["state_id"]
                    for record in records
                    if record["split"] == name
                ),
            }
            for name in SPLIT_NAMES
        }

        primitive_steps = max(HORIZONS) * FRAMESKIP
        first = records[0]
        first_task = tasks_by_id[first["task_id"]]
        first_actions, labels, selected = candidate_library(
            environment,
            first["state"],
            first_task,
            primitive_steps,
        )
        restore_payload[environment] = exact_restore_test(
            repo,
            environment,
            first_task,
            first["state"],
            first_actions[1],
        )

        state_matrix = []
        task_ids = []
        action_bank = []
        physical_costs = []
        interaction_bank = []
        interaction_type_bank = []
        for record in records:
            state_id = record["state_id"]
            state_path = truth_dir / f"state_{state_id:04d}.npz"
            if state_path.exists():
                log.info("%s simulator resume: keeping %s", environment, state_path.name)
                with np.load(state_path) as shard:
                    state_matrix.append(shard["initial_state"])
                    task_ids.append(int(shard["task_id"]))
                    action_bank.append(shard["selected_actions"])
                    physical_costs.append(shard["physical_cost"])
                    interaction_bank.append(shard["interactions"])
                    interaction_type_bank.append(shard["interaction_types"])
                continue

            task = tasks_by_id[record["task_id"]]
            actions, action_labels, selected_indices = candidate_library(
                environment,
                record["state"],
                task,
                primitive_steps,
            )
            if action_labels != labels:
                raise AssertionError("candidate labels changed across states")
            initials = []
            visuals = []
            proprios = []
            endpoints = []
            all_visuals = []
            all_proprios = []
            all_endpoints = []
            interactions = []
            interaction_types = []
            for branch in actions:
                (
                    initial,
                    _,
                    observations,
                    states,
                    counts,
                    kinds,
                ) = rollout_branch(
                    repo,
                    environment,
                    task,
                    record["state"],
                    branch,
                    record["evaluation_seed"] * 1000 + state_id,
                )
                initials.append(initial["visual"])
                visuals.append(
                    [observations[horizon]["visual"] for horizon in HORIZONS]
                )
                proprios.append(
                    [observations[horizon]["proprio"] for horizon in HORIZONS]
                )
                endpoints.append(
                    [states[horizon] for horizon in HORIZONS]
                )
                all_visuals.append(
                    [observations[step]["visual"] for step in TARGET_STEPS]
                )
                all_proprios.append(
                    [observations[step]["proprio"] for step in TARGET_STEPS]
                )
                all_endpoints.append(
                    [states[step] for step in TARGET_STEPS]
                )
                interactions.append(
                    [counts[horizon] for horizon in HORIZONS]
                )
                interaction_types.append(
                    [kinds[horizon] for horizon in HORIZONS]
                )
            if not all(
                np.array_equal(initials[0], item) for item in initials[1:]
            ):
                raise AssertionError(
                    f"branch initial render mismatch: {environment} state {state_id}"
                )
            endpoint_array = np.asarray(endpoints, dtype=np.float32)
            physical_cost = task_cost(
                environment, endpoint_array, task
            ).astype(np.float32)
            _, initial_observation, _ = reset_environment(
                repo,
                environment,
                task,
                record["state"],
                record["evaluation_seed"] * 1000 + state_id,
            )
            atomic_npz(
                state_path,
                initial_state=np.asarray(record["state"], dtype=np.float64),
                task_id=np.asarray(record["task_id"], dtype=np.int64),
                task_split=np.asarray(record["split"]),
                evaluation_seed=np.asarray(
                    record["evaluation_seed"], dtype=np.int64
                ),
                design_stratum=np.asarray(record["design_stratum"]),
                initial_visual=initials[0],
                initial_proprio=initial_observation["proprio"],
                selected_actions=actions,
                selected_library_indices=selected_indices,
                action_labels=np.asarray(action_labels),
                future_visual=np.asarray(visuals, dtype=np.uint8),
                future_proprio=np.asarray(proprios, dtype=np.float32),
                endpoint_states=endpoint_array,
                all_future_visual=np.asarray(all_visuals, dtype=np.uint8),
                all_future_proprio=np.asarray(all_proprios, dtype=np.float32),
                all_endpoint_states=np.asarray(all_endpoints, dtype=np.float32),
                physical_cost=physical_cost,
                interactions=np.asarray(interactions, dtype=np.int32),
                interaction_types=np.asarray(interaction_types),
            )
            state_matrix.append(record["state"])
            task_ids.append(record["task_id"])
            action_bank.append(actions)
            physical_costs.append(physical_cost)
            interaction_bank.append(interactions)
            interaction_type_bank.append(interaction_types)
            write_json(
                OUT / f"{environment.lower()}_simulator_progress.json",
                {
                    "run_signature": RUN_SIGNATURE,
                    "environment": environment,
                    "completed_states": state_id + 1,
                    "total_states": NUM_STATES,
                    "last_file": state_path.name,
                },
            )
            log.info(
                "%s simulator state %d/%d",
                environment,
                state_id + 1,
                NUM_STATES,
            )

        physical_costs = np.asarray(physical_costs, dtype=np.float64)
        interactions = np.asarray(interaction_bank, dtype=np.int32)
        oracle = np.argmin(physical_costs, axis=1)
        spread = np.max(physical_costs, axis=1) - np.min(
            physical_costs, axis=1
        )
        no_op_regret = physical_costs[:, 0] - np.min(
            physical_costs, axis=1
        )
        left, right = pair_indices(ACTIONS_PER_STATE)
        pair_interactions = (
            (interactions[:, left, :] > 0).astype(int)
            + (interactions[:, right, :] > 0).astype(int)
        )
        environment_design = {
            "selection_protocol": (
                "fixed state/task-relative candidates; future simulator outcomes "
                "are never used for candidate selection"
            ),
            "candidate_labels": labels,
            "no_op_oracle_fraction_by_horizon": np.mean(
                oracle == 0, axis=0
            ).tolist(),
            "no_op_positive_regret_fraction_by_horizon": np.mean(
                no_op_regret > 1e-9, axis=0
            ).tolist(),
            "median_physical_cost_spread_by_horizon": np.median(
                spread, axis=0
            ).tolist(),
            "minimum_physical_cost_spread_by_horizon": np.min(
                spread, axis=0
            ).tolist(),
            "interaction_fraction_by_horizon": np.mean(
                interactions > 0, axis=(0, 1)
            ).tolist(),
            "pair_interaction_counts": {
                label: int(np.sum(pair_interactions == index))
                for index, label in enumerate(["neither", "one", "both"])
            },
        }
        environment_design["validity_thresholds"] = {
            "final_horizon_no_op_oracle_fraction_max": 0.25,
            "final_horizon_no_op_positive_regret_fraction_min": 0.75,
            "final_horizon_median_cost_spread_min": (
                0.08 if environment == "PushT" else 0.05
            ),
            "all_pair_interaction_strata_required": True,
        }
        environment_design["design_valid"] = bool(
            environment_design[
                "no_op_oracle_fraction_by_horizon"
            ][-1]
            < 0.25
            and environment_design[
                "no_op_positive_regret_fraction_by_horizon"
            ][-1]
            > 0.75
            and environment_design[
                "median_physical_cost_spread_by_horizon"
            ][-1]
            > (0.08 if environment == "PushT" else 0.05)
            and all(
                environment_design["pair_interaction_counts"][label] > 0
                for label in ["neither", "one", "both"]
            )
        )
        design_payload[environment] = environment_design
        atomic_npz(
            OUT / f"{environment.lower()}_design.npz",
            states=np.asarray(state_matrix),
            task_ids=np.asarray(task_ids),
            action_bank=np.asarray(action_bank, dtype=np.float32),
            physical_cost=physical_costs.astype(np.float32),
            interactions=interactions,
            interaction_types=np.asarray(interaction_type_bank),
            candidate_labels=np.asarray(labels),
        )

    write_json(OUT / "tasks.json", task_payload)
    write_json(OUT / "split_manifest.json", split_payload)
    write_json(OUT / "restore_test.json", restore_payload)
    write_json(OUT / "candidate_design_summary.json", design_payload)
    return repo


if not PIPELINE_FAILED:
    try:
        REPO = generate_simulator_truth()
    except Exception:
        record_failure("simulator_truth")

In [ ]:
# Phase B — cache or reuse frozen JEPA-WM transitions and all audited layers.

TRANSITION_ROOT = INTERMEDIATE / "transitions"
GOAL_ROOT = INTERMEDIATE / "goals"
for path in [TRANSITION_ROOT, GOAL_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

AUDIT_LAYERS = [f"block_{index + 1}" for index in range(6)] + [
    "predictor_output"
]


def atomic_npz_uncompressed(path, **arrays):
    path = Path(path)
    temporary = path.with_suffix(".tmp.npz")
    np.savez(temporary, **arrays)
    temporary.replace(path)


def visual_tokens(value):
    """Return [batch, tokens, channels] without spatial pooling."""
    if hasattr(value, "detach"):
        value = value.detach().float().cpu().numpy()
    value = np.asarray(value)
    if value.ndim == 6:
        # [B, T, V, H, W, D] -> last time step.
        value = value[:, -1, 0]
    elif value.ndim == 5:
        # [B, V, H, W, D].
        value = value[:, 0]
    elif value.ndim != 3:
        raise ValueError(f"unexpected visual-token shape {value.shape}")
    return value.reshape(value.shape[0], -1, value.shape[-1])


class CountSketchProjector:
    """Deterministic memory-light projection of flattened patch deltas."""

    def __init__(self, input_dim, output_dim, seed, device="cuda"):
        rng = np.random.default_rng(seed)
        bucket = rng.integers(0, output_dim, size=input_dim, dtype=np.int64)
        sign = rng.choice(np.asarray([-1.0, 1.0], dtype=np.float32), input_dim)
        self.bucket = torch.as_tensor(bucket, device=device, dtype=torch.long)
        self.sign = torch.as_tensor(sign, device=device, dtype=torch.float32)
        counts = np.bincount(bucket, minlength=output_dim).astype(np.float32)
        counts[counts == 0] = 1.0
        self.scale = torch.as_tensor(
            np.sqrt(counts), device=device, dtype=torch.float32
        )
        self.output_dim = int(output_dim)

    def __call__(self, values):
        values = values.float().flatten(1)
        output = torch.zeros(
            values.shape[0],
            self.output_dim,
            device=values.device,
            dtype=torch.float32,
        )
        output.scatter_add_(
            1,
            self.bucket[None].expand(values.shape[0], -1),
            values * self.sign[None],
        )
        return output / self.scale[None]


def model_action_tensor(preprocessor, selected_actions):
    chunks = torch.from_numpy(
        selected_actions.reshape(
            ACTIONS_PER_STATE,
            max(HORIZONS),
            FRAMESKIP,
            2,
        )
    ).float()
    normalized = preprocessor.normalize_actions(chunks)
    return (
        normalized.reshape(ACTIONS_PER_STATE, max(HORIZONS), -1)
        .permute(1, 0, 2)
        .contiguous()
        .cuda()
    )


def validate_jepa_predictor(model, model_name):
    predictor = model.model.predictor
    blocks = list(getattr(predictor, "predictor_blocks", []))
    if len(blocks) != 6:
        raise RuntimeError(
            f"{model_name} expected six AdaLN blocks, found {len(blocks)}"
        )
    if predictor.__class__.__name__ != "VisionTransformerAdaLN":
        raise RuntimeError(
            f"{model_name} is not the expected AdaLN predictor: "
            f"{predictor.__class__.__name__}"
        )
    if not bool(getattr(predictor, "action_encoder_inpred", False)):
        raise RuntimeError(f"{model_name} does not encode actions in predictor")
    return predictor, blocks


def encode_all_true_tokens(model, future_visual, future_proprio):
    encoded = model.encode(
        to_model_observation(future_visual, future_proprio)
    )
    visual = encoded["visual"].detach().float().cpu().numpy()
    # [A, S, 1, H, W, D] -> [A, S, H*W, D]
    return visual[:, :, 0].reshape(
        visual.shape[0], visual.shape[1], -1, visual.shape[-1]
    )


def layer_tokens_from_capture(capture, batch, time_steps, visual_dim):
    value = capture
    if value.ndim != 3:
        raise ValueError(f"unexpected AdaLN block output {tuple(value.shape)}")
    tokens_per_step = value.shape[1] // time_steps
    if tokens_per_step != 16 * 16:
        raise ValueError(
            f"unexpected tokens per step {tokens_per_step}; expected 256"
        )
    return value.view(
        batch, time_steps, tokens_per_step, value.shape[-1]
    )[:, -1, :, :visual_dim]


def base_unroll_with_layer_audit(model, initial_encoded, model_actions):
    predictor, blocks = validate_jepa_predictor(model, "loaded_model")
    visual_dim = int(predictor.predictor_embed_dim)
    projection_input_dim = 16 * 16 * visual_dim
    projectors = [
        CountSketchProjector(
            projection_input_dim,
            AUDIT_PROJECTION_DIM,
            seed,
        )
        for seed in AUDIT_PROJECTION_SEEDS
    ]

    captures = []
    handles = []
    for block in blocks:
        handles.append(
            block.register_forward_hook(
                lambda _module, _inputs, output: captures.append(output)
            )
        )

    try:
        action_batch = model_actions.permute(1, 0, 2).contiguous()
        action_features = model.model.encode_act(action_batch)
        visual_history = initial_encoded["visual"].expand(
            ACTIONS_PER_STATE, *initial_encoded["visual"].shape[1:]
        )
        proprio_history = initial_encoded["proprio"].expand(
            ACTIONS_PER_STATE, *initial_encoded["proprio"].shape[1:]
        )

        base_predictions = []
        base_deltas = []
        base_proprios = []
        audit_steps = []

        for step_index in range(max(HORIZONS)):
            captures.clear()
            action_prefix = action_features[:, : step_index + 1]
            predicted_visual, _, predicted_proprio = model.model.forward_pred(
                visual_history[:, -model.ctxt_window :],
                action_prefix[:, -model.ctxt_window :],
                proprio_history[:, -model.ctxt_window :],
            )
            if len(captures) != len(blocks):
                raise RuntimeError(
                    f"captured {len(captures)} blocks, expected {len(blocks)}"
                )
            next_visual = predicted_visual[:, -1:]
            next_proprio = predicted_proprio[:, -1:]
            current_visual = visual_history[:, -1:]
            next_tokens = next_visual[:, 0, 0].flatten(1, 2)
            current_tokens = current_visual[:, 0, 0].flatten(1, 2)
            base_predictions.append(next_tokens.detach().cpu().numpy())
            base_deltas.append(
                (next_tokens - current_tokens).detach().cpu().numpy()
            )
            base_proprios.append(
                next_proprio[:, 0].detach().float().cpu().numpy()
            )

            time_steps = predicted_visual.shape[1]
            layer_values = [
                layer_tokens_from_capture(
                    captured,
                    ACTIONS_PER_STATE,
                    time_steps,
                    visual_dim,
                )
                for captured in captures
            ]
            layer_values.append(next_tokens)
            per_seed = []
            for projector in projectors:
                per_layer = []
                for value in layer_values:
                    effect = value - value[:1]
                    per_layer.append(projector(effect).detach().cpu().numpy())
                per_seed.append(np.stack(per_layer, axis=0))
            audit_steps.append(np.stack(per_seed, axis=0))

            visual_history = torch.cat(
                [visual_history, next_visual], dim=1
            )
            proprio_history = torch.cat(
                [proprio_history, next_proprio], dim=1
            )

        return {
            "base_prediction": np.stack(base_predictions, axis=1),
            "base_delta": np.stack(base_deltas, axis=1),
            "base_proprio": np.stack(base_proprios, axis=1),
            # [step, projection_seed, layer, action, projection]
            "audit_projection": np.stack(audit_steps, axis=0),
            "normalized_action": action_batch.detach().cpu().numpy(),
        }
    finally:
        for handle in handles:
            handle.remove()


def cache_transition_tokens():
    repo = configure_repo()
    checkpoint_records = []

    for environment in ENVIRONMENT:
        truth_dir = TRUTH_ROOT / environment.lower()
        for model_name in MODEL_BY_ENVIRONMENT[environment]:
            transition_dir = TRANSITION_ROOT / model_name
            transition_dir.mkdir(parents=True, exist_ok=True)
            goal_path = GOAL_ROOT / f"{model_name}.npz"

            torch.cuda.reset_peak_memory_stats()
            gpu_report(f"{model_name}_before_load")
            model, preprocessor = torch.hub.load(
                str(repo),
                model_name,
                source="local",
                pretrained=True,
                device="cuda:0",
                trust_repo=True,
            )
            model.eval()
            predictor, _ = validate_jepa_predictor(model, model_name)
            gpu_report(f"{model_name}_after_load")

            if not goal_path.exists():
                goal_visual = []
                goal_proprio = []
                with torch.inference_mode():
                    for task in TASKS[environment]:
                        observation = goal_observation(repo, environment, task)
                        encoded = model.encode(
                            to_model_observation(
                                observation["visual"],
                                observation["proprio"],
                            )
                        )
                        goal_visual.append(
                            encoded["visual"][0, -1, 0]
                            .flatten(0, 1)
                            .detach()
                            .float()
                            .cpu()
                            .numpy()
                        )
                        goal_proprio.append(
                            encoded["proprio"][0, -1]
                            .detach()
                            .float()
                            .cpu()
                            .numpy()
                        )
                atomic_npz_uncompressed(
                    goal_path,
                    visual=np.asarray(goal_visual, dtype=np.float16),
                    proprio=np.asarray(goal_proprio, dtype=np.float16),
                )

            for state_id in range(NUM_STATES):
                output_path = transition_dir / f"state_{state_id:04d}.npz"
                if output_path.exists():
                    log.info(
                        "%s transition resume: keeping %s",
                        model_name,
                        output_path.name,
                    )
                    continue
                with np.load(
                    truth_dir / f"state_{state_id:04d}.npz"
                ) as truth:
                    initial_visual = truth["initial_visual"]
                    initial_proprio = truth["initial_proprio"]
                    all_future_visual = truth["all_future_visual"]
                    all_future_proprio = truth["all_future_proprio"]
                    selected_actions = truth["selected_actions"]
                    task_id = int(truth["task_id"])

                actions = model_action_tensor(
                    preprocessor, selected_actions
                )
                with torch.inference_mode():
                    initial_encoded = model.encode(
                        to_model_observation(
                            initial_visual,
                            initial_proprio,
                        )
                    )
                    true_tokens = encode_all_true_tokens(
                        model,
                        all_future_visual,
                        all_future_proprio,
                    )
                    cached = base_unroll_with_layer_audit(
                        model,
                        initial_encoded,
                        actions,
                    )

                atomic_npz_uncompressed(
                    output_path,
                    task_id=np.asarray(task_id, dtype=np.int64),
                    true_tokens=true_tokens.astype(np.float16),
                    base_prediction=cached["base_prediction"].astype(
                        np.float16
                    ),
                    base_delta=cached["base_delta"].astype(np.float16),
                    base_proprio=cached["base_proprio"].astype(np.float16),
                    normalized_action=cached["normalized_action"].astype(
                        np.float32
                    ),
                    audit_projection=cached["audit_projection"].astype(
                        np.float16
                    ),
                )
                write_json(
                    OUT / f"{model_name}_transition_progress.json",
                    {
                        "run_signature": RUN_SIGNATURE,
                        "environment": environment,
                        "model": model_name,
                        "completed_states": state_id + 1,
                        "total_states": NUM_STATES,
                        "last_file": output_path.name,
                    },
                )
                log.info(
                    "%s transition state %d/%d",
                    model_name,
                    state_id + 1,
                    NUM_STATES,
                )
                if (state_id + 1) % 12 == 0:
                    gpu_report(
                        f"{model_name}_transition_state_{state_id:04d}"
                    )

            del model, preprocessor, predictor
            gc.collect()
            torch.cuda.empty_cache()
            gpu_report(f"{model_name}_released")

    hf_root = Path(os.environ["HF_HOME"]) / "hub"
    torch_root = Path(os.environ["TORCH_HOME"])
    for root in [hf_root, torch_root]:
        if root.exists():
            for path in root.rglob("*"):
                if (
                    path.is_file()
                    and path.stat().st_size > 20_000_000
                    and path.suffix
                    in {".tar", ".pth", ".pt", ".bin", ".safetensors"}
                ):
                    checkpoint_records.append(
                        {
                            "path": str(path),
                            "size_bytes": path.stat().st_size,
                            "sha256": sha256_file(path),
                        }
                    )
    write_json(
        OUT / "checkpoints_manifest.json",
        {
            "models": MODEL_NAME,
            "repository": "facebook/jepa-wms",
            "repository_commit": REPO_COMMIT,
            "predictor_requirement": "six-block VisionTransformerAdaLN",
            "cached_files": checkpoint_records,
        },
    )


if not PIPELINE_FAILED:
    try:
        cache_transition_tokens()
    except Exception:
        record_failure("transition_token_cache_and_layer_audit")

In [ ]:
# Phase C — adapt the causal action-conditioning path.

ADAPTED_DIR = OUT / "adapted_action_paths"
ADAPTATION_FEATURE_DIR = INTERMEDIATE / "stage9_rollout_features"
for path in [ADAPTED_DIR, ADAPTATION_FEATURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CONFIG.update(
    {
        "ADAPTATION_METHODS": ADAPTATION_METHODS,
        "ADAPTATION_SEED": ADAPTATION_SEED,
        "ADAPTATION_EPOCHS": ADAPTATION_EPOCHS,
        "ADAPTATION_LR": ADAPTATION_LR,
        "ADAPTATION_WEIGHT_DECAY": ADAPTATION_WEIGHT_DECAY,
        "GRADIENT_CLIP": GRADIENT_CLIP,
        "READOUT_PROJECTION_DIM": READOUT_PROJECTION_DIM,
        "EVALUATION_PROJECTION_SEED": EVALUATION_PROJECTION_SEED,
        "ACTION_GROUP_SIZE": ACTION_GROUP_SIZE,
        "LOSS_WEIGHTS": LOSS_WEIGHTS,
        "USE_ACTIVATION_CHECKPOINTING": USE_ACTIVATION_CHECKPOINTING,
        "EVIDENCE_STATUS": EVIDENCE_STATUS,
        "TASK_FAMILY_ID": TASK_FAMILY_ID,
    }
)
write_json(OUT / "config.json", CONFIG)


class TemporarySemanticHead(torch.nn.Module):
    def __init__(self, input_dim, pose_dim, action_dim):
        super().__init__()
        self.pose = torch.nn.Linear(input_dim, pose_dim)
        self.action = torch.nn.Linear(input_dim, action_dim)


def action_path_named_parameters(predictor):
    named = []
    for name, parameter in predictor.action_encoder.named_parameters():
        named.append((f"action_encoder.{name}", parameter))
    for index, block in enumerate(predictor.predictor_blocks):
        for name, parameter in block.adaLN_modulation[1].named_parameters():
            named.append(
                (f"predictor_blocks.{index}.adaLN_modulation.1.{name}", parameter)
            )
    return named


def extract_action_path_state(predictor):
    return {
        name: parameter.detach().cpu().clone()
        for name, parameter in action_path_named_parameters(predictor)
    }


def load_action_path_state(predictor, state):
    named = dict(action_path_named_parameters(predictor))
    if set(named) != set(state):
        raise RuntimeError(
            f"action-path keys differ: model={sorted(named)} state={sorted(state)}"
        )
    with torch.no_grad():
        for name, parameter in named.items():
            parameter.copy_(state[name].to(parameter.device, parameter.dtype))


def set_action_path_trainability(model, trainable):
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    predictor, _ = validate_jepa_predictor(model, "loaded_model")
    for _, parameter in action_path_named_parameters(predictor):
        parameter.requires_grad_(trainable)
    return predictor


def state_ids_for_split(environment, split_name):
    return [
        record["state_id"]
        for record in build_state_records(environment)
        if record["split"] == split_name
    ]


def deterministic_outcome_permutation(state_id):
    rng = np.random.default_rng(
        ADAPTATION_SEED + 1009 * int(state_id)
    )
    non_null = np.arange(1, ACTIONS_PER_STATE, dtype=np.int64)
    rng.shuffle(non_null)
    return np.concatenate(
        [np.asarray([0], dtype=np.int64), non_null]
    )


def action_groups():
    non_null = list(range(1, ACTIONS_PER_STATE))
    width = ACTION_GROUP_SIZE - 1
    return [
        np.asarray([0, *non_null[start : start + width]], dtype=np.int64)
        for start in range(0, len(non_null), width)
    ]


def pose_statistics(environment, state_ids):
    values = []
    truth_dir = TRUTH_ROOT / environment.lower()
    for state_id in state_ids:
        with np.load(truth_dir / f"state_{state_id:04d}.npz") as shard:
            values.append(
                pose_target(environment, shard["all_endpoint_states"])
            )
    values = np.concatenate(
        [value.reshape(-1, value.shape[-1]) for value in values],
        axis=0,
    )
    mean = values.mean(axis=0).astype(np.float32)
    scale = values.std(axis=0).astype(np.float32)
    scale[scale < 1e-4] = 1.0
    return mean, scale


def cache_initial_encodings(model, environment, state_ids):
    cache = {}
    truth_dir = TRUTH_ROOT / environment.lower()
    with torch.inference_mode():
        for state_id in state_ids:
            with np.load(
                truth_dir / f"state_{state_id:04d}.npz"
            ) as shard:
                encoded = model.encode(
                    to_model_observation(
                        shard["initial_visual"],
                        shard["initial_proprio"],
                    )
                )
            cache[state_id] = {
                key: value.detach().to("cpu", dtype=torch.float16)
                for key, value in encoded.items()
            }
    return cache


def initial_to_cuda(initial):
    return {
        key: value.to("cuda", dtype=torch.float32, non_blocking=True)
        for key, value in initial.items()
    }


def differentiable_unroll(model, initial_encoded, action_batch):
    batch = action_batch.shape[0]
    action_features = model.model.encode_act(action_batch)
    visual_history = initial_encoded["visual"].expand(
        batch, *initial_encoded["visual"].shape[1:]
    )
    proprio_history = initial_encoded["proprio"].expand(
        batch, *initial_encoded["proprio"].shape[1:]
    )
    predictions = []
    displacements = []
    proprios = []
    for step_index in range(max(HORIZONS)):
        predicted_visual, _, predicted_proprio = model.model.forward_pred(
            visual_history[:, -model.ctxt_window :],
            action_features[:, : step_index + 1][
                :, -model.ctxt_window :
            ],
            proprio_history[:, -model.ctxt_window :],
        )
        next_visual = predicted_visual[:, -1:]
        next_proprio = predicted_proprio[:, -1:]
        current_visual = visual_history[:, -1:]
        next_tokens = next_visual[:, 0, 0].flatten(1, 2)
        current_tokens = current_visual[:, 0, 0].flatten(1, 2)
        predictions.append(next_tokens)
        displacements.append(next_tokens - current_tokens)
        proprios.append(next_proprio[:, 0])
        visual_history = torch.cat([visual_history, next_visual], dim=1)
        proprio_history = torch.cat(
            [proprio_history, next_proprio], dim=1
        )
    return (
        torch.stack(predictions, dim=1),
        torch.stack(displacements, dim=1),
        torch.stack(proprios, dim=1),
    )


def load_adaptation_state(environment, model_name, state_id):
    truth_path = (
        TRUTH_ROOT
        / environment.lower()
        / f"state_{state_id:04d}.npz"
    )
    transition_path = (
        TRANSITION_ROOT
        / model_name
        / f"state_{state_id:04d}.npz"
    )
    with np.load(truth_path) as truth:
        pose = pose_target(
            environment, truth["all_endpoint_states"]
        ).astype(np.float32)
    with np.load(transition_path) as transition:
        actions = transition["normalized_action"].astype(np.float32)
        true_tokens = transition["true_tokens"].astype(np.float32)
    return pose, actions, true_tokens


def normalized_pose_tensor(pose, mean, scale):
    return torch.as_tensor(
        (pose - mean[None, None]) / scale[None, None],
        device="cuda",
        dtype=torch.float32,
    )


def proximity_loss(predictor, reference):
    pieces = []
    for name, parameter in action_path_named_parameters(predictor):
        target = reference[name].to(parameter.device, parameter.dtype)
        denominator = target.square().mean().clamp_min(1e-6)
        pieces.append((parameter - target).square().mean() / denominator)
    return torch.stack(pieces).mean()


def adaptation_losses(
    method,
    predicted_tokens,
    displacements,
    true_tokens,
    target_pose,
    action_target,
    head,
    projector,
    predictor,
    reference,
):
    anchor = torch_functional.smooth_l1_loss(
        predicted_tokens, true_tokens
    )
    zero = anchor.new_zeros(())
    state = zero
    effect = zero
    action = zero
    if method != "latent_only_action_path":
        batch, steps, tokens, channels = predicted_tokens.shape
        projected = projector(
            predicted_tokens.reshape(batch * steps, tokens, channels)
        ).reshape(batch, steps, -1)
        projected_delta = projector(
            displacements.reshape(batch * steps, tokens, channels)
        ).reshape(batch, steps, -1)
        predicted_pose = head.pose(projected)
        predicted_action = head.action(projected_delta)
        state = torch_functional.smooth_l1_loss(
            predicted_pose, target_pose
        )
        effect = torch_functional.smooth_l1_loss(
            predicted_pose - predicted_pose[:1],
            target_pose - target_pose[:1],
        )
        action = torch_functional.mse_loss(
            predicted_action, action_target
        )
    proximity = proximity_loss(predictor, reference)
    total = (
        LOSS_WEIGHTS["anchor"] * anchor
        + LOSS_WEIGHTS["state"] * state
        + LOSS_WEIGHTS["effect"] * effect
        + LOSS_WEIGHTS["action"] * action
        + LOSS_WEIGHTS["proximity"] * proximity
    )
    return total, {
        "anchor": float(anchor.detach()),
        "state": float(state.detach()),
        "effect": float(effect.detach()),
        "action": float(action.detach()),
        "proximity": float(proximity.detach()),
        "total": float(total.detach()),
    }


def calibration_objective(
    model,
    method,
    environment,
    model_name,
    state_ids,
    initial_cache,
    projector,
    head,
    pose_mean,
    pose_scale,
):
    losses = []
    with torch.inference_mode():
        for state_id in state_ids:
            pose, actions, true_tokens = load_adaptation_state(
                environment, model_name, state_id
            )
            if method == "shuffled_cave_action_path":
                pose = pose[deterministic_outcome_permutation(state_id)]
            initial = initial_to_cuda(initial_cache[state_id])
            action_tensor = torch.as_tensor(
                actions, device="cuda", dtype=torch.float32
            )
            predicted, displacement, _ = differentiable_unroll(
                model, initial, action_tensor
            )
            true_tensor = torch.as_tensor(
                true_tokens, device="cuda", dtype=torch.float32
            )
            anchor = torch_functional.smooth_l1_loss(
                predicted, true_tensor
            )
            if method == "latent_only_action_path":
                losses.append(float(anchor))
                continue
            target = normalized_pose_tensor(
                pose, pose_mean, pose_scale
            )
            batch, steps, tokens, channels = predicted.shape
            projected = projector(
                predicted.reshape(batch * steps, tokens, channels)
            ).reshape(batch, steps, -1)
            predicted_pose = head.pose(projected)
            state = torch_functional.smooth_l1_loss(
                predicted_pose, target
            )
            effect = torch_functional.smooth_l1_loss(
                predicted_pose - predicted_pose[:1],
                target - target[:1],
            )
            losses.append(
                float(
                    LOSS_WEIGHTS["anchor"] * anchor
                    + LOSS_WEIGHTS["state"] * state
                    + LOSS_WEIGHTS["effect"] * effect
                )
            )
    return float(np.mean(losses))


def train_one_action_path(
    model,
    environment,
    model_name,
    method,
    base_state,
    train_ids,
    calibration_ids,
    initial_cache,
    pose_mean,
    pose_scale,
):
    output_path = ADAPTED_DIR / f"{model_name}_{method}.pt"
    if output_path.exists():
        log.info("adaptation resume: keeping %s", output_path.name)
        return torch.load(output_path, map_location="cpu", weights_only=False)

    predictor = set_action_path_trainability(model, True)
    load_action_path_state(predictor, base_state)
    predictor.use_activation_checkpointing = USE_ACTIVATION_CHECKPOINTING
    pose_dim = 4 if environment == "PushT" else 2
    action_dim = int(
        np.load(
            TRANSITION_ROOT / model_name / "state_0000.npz"
        )["normalized_action"].shape[-1]
    )
    torch.manual_seed(
        ADAPTATION_SEED + 1000 * ENVIRONMENT.index(environment)
        + 100 * ADAPTATION_METHODS.index(method)
    )
    torch.cuda.manual_seed_all(
        ADAPTATION_SEED + 1000 * ENVIRONMENT.index(environment)
        + 100 * ADAPTATION_METHODS.index(method)
    )
    head = TemporarySemanticHead(
        READOUT_PROJECTION_DIM, pose_dim, action_dim
    ).cuda()
    head.apply(
        lambda module: (
            torch.nn.init.xavier_uniform_(module.weight)
            if isinstance(module, torch.nn.Linear)
            else None
        )
    )
    projector = CountSketchProjector(
        16 * 16 * int(predictor.predictor_embed_dim),
        READOUT_PROJECTION_DIM,
        ADAPTATION_SEED + 17 * ENVIRONMENT.index(environment),
    )
    parameters = [
        parameter
        for _, parameter in action_path_named_parameters(predictor)
    ]
    if method != "latent_only_action_path":
        parameters += list(head.parameters())
    optimizer = torch.optim.AdamW(
        parameters,
        lr=ADAPTATION_LR,
        weight_decay=ADAPTATION_WEIGHT_DECAY,
    )
    rng = np.random.default_rng(
        ADAPTATION_SEED + 31 * ADAPTATION_METHODS.index(method)
    )
    history = []
    best = None

    for epoch in range(1, ADAPTATION_EPOCHS + 1):
        order = np.asarray(train_ids, dtype=np.int64).copy()
        rng.shuffle(order)
        epoch_terms = []
        model.eval()
        for state_id in order:
            pose, actions, true_tokens = load_adaptation_state(
                environment, model_name, int(state_id)
            )
            if method == "shuffled_cave_action_path":
                pose = pose[
                    deterministic_outcome_permutation(int(state_id))
                ]
            initial = initial_to_cuda(initial_cache[int(state_id)])
            for group in action_groups():
                action_tensor = torch.as_tensor(
                    actions[group], device="cuda", dtype=torch.float32
                )
                target_tokens = torch.as_tensor(
                    true_tokens[group],
                    device="cuda",
                    dtype=torch.float32,
                )
                target_pose = normalized_pose_tensor(
                    pose[group], pose_mean, pose_scale
                )
                optimizer.zero_grad(set_to_none=True)
                predicted, displacement, _ = differentiable_unroll(
                    model, initial, action_tensor
                )
                total, terms = adaptation_losses(
                    method,
                    predicted,
                    displacement,
                    target_tokens,
                    target_pose,
                    action_tensor,
                    head,
                    projector,
                    predictor,
                    base_state,
                )
                total.backward()
                torch.nn.utils.clip_grad_norm_(parameters, GRADIENT_CLIP)
                optimizer.step()
                epoch_terms.append(terms)

        row = {
            "environment": environment,
            "model": model_name,
            "method": method,
            "epoch": epoch,
        }
        for key in ["anchor", "state", "effect", "action", "proximity", "total"]:
            row[key] = float(np.mean([item[key] for item in epoch_terms]))

        if epoch in SELECTION_EPOCHS:
            predictor.use_activation_checkpointing = False
            score = calibration_objective(
                model,
                method,
                environment,
                model_name,
                calibration_ids,
                initial_cache,
                projector,
                head,
                pose_mean,
                pose_scale,
            )
            predictor.use_activation_checkpointing = USE_ACTIVATION_CHECKPOINTING
            row["calibration_objective"] = score
            candidate = {
                "method": method,
                "environment": environment,
                "model": model_name,
                "selected_epoch": epoch,
                "calibration_objective": score,
                "action_path": extract_action_path_state(predictor),
                "pose_mean": pose_mean,
                "pose_scale": pose_scale,
                "config": {
                    "loss_weights": LOSS_WEIGHTS,
                    "lr": ADAPTATION_LR,
                    "weight_decay": ADAPTATION_WEIGHT_DECAY,
                    "seed": ADAPTATION_SEED,
                },
            }
            if best is None or score < best["calibration_objective"]:
                best = candidate
        else:
            row["calibration_objective"] = float("nan")
        history.append(row)
        log.info(
            "%s %s epoch=%d total=%.6f calibration=%s",
            model_name,
            method,
            epoch,
            row["total"],
            row["calibration_objective"],
        )

    if best is None:
        raise RuntimeError("no adaptation checkpoint was eligible")
    torch.save(best, output_path)
    existing = []
    history_path = OUT / "action_path_training_history.csv"
    if history_path.exists():
        with history_path.open() as handle:
            existing = list(csv.DictReader(handle))
    write_csv(history_path, [*existing, *history])
    load_action_path_state(predictor, best["action_path"])
    predictor.use_activation_checkpointing = False
    return best


def adapt_action_paths():
    manifest = []
    repo = configure_repo()
    for environment in ENVIRONMENT:
        model_name = MODEL_BY_ENVIRONMENT[environment][0]
        model, _ = torch.hub.load(
            str(repo),
            model_name,
            source="local",
            pretrained=True,
            device="cuda:0",
            trust_repo=True,
        )
        model.eval()
        predictor = set_action_path_trainability(model, False)
        base_state = extract_action_path_state(predictor)
        train_ids = state_ids_for_split(environment, "probe_train")
        calibration_ids = state_ids_for_split(
            environment, "probe_calibration"
        )
        initial_cache = cache_initial_encodings(
            model, environment, [*train_ids, *calibration_ids]
        )
        pose_mean, pose_scale = pose_statistics(environment, train_ids)
        for method in ADAPTATION_METHODS:
            result = train_one_action_path(
                model,
                environment,
                model_name,
                method,
                base_state,
                train_ids,
                calibration_ids,
                initial_cache,
                pose_mean,
                pose_scale,
            )
            manifest.append(
                {
                    "environment": environment,
                    "model": model_name,
                    "method": method,
                    "selected_epoch": result["selected_epoch"],
                    "calibration_objective": result[
                        "calibration_objective"
                    ],
                    "trainable_parameter_count": int(
                        sum(
                            parameter.numel()
                            for _, parameter in action_path_named_parameters(
                                predictor
                            )
                        )
                    ),
                }
            )
        del model, predictor, initial_cache
        gc.collect()
        torch.cuda.empty_cache()
    write_json(
        OUT / "action_path_adaptation_manifest.json",
        {
            "run_signature": RUN_SIGNATURE,
            "updated_modules": [
                "predictor.action_encoder",
                "predictor.predictor_blocks.*.adaLN_modulation[1]",
            ],
            "frozen_modules": [
                "visual encoder",
                "predictor attention and MLP content weights",
                "predictor output projection",
                "proprio path",
            ],
            "records": manifest,
        },
    )


if not PIPELINE_FAILED:
    try:
        adapt_action_paths()
    except Exception:
        record_failure("counterfactual_value_equivalent_action_path_adaptation")


In [ ]:
# Phase D — discard training heads; refit identical linear readouts and evaluate.

EVALUATION_VARIANTS = ["frozen", *ADAPTATION_METHODS]
HORIZON_INDICES = [horizon - 1 for horizon in HORIZONS]


def rollout_feature_path(model_name, variant, state_id):
    return (
        ADAPTATION_FEATURE_DIR
        / model_name
        / variant
        / f"state_{state_id:04d}.npz"
    )


def cache_variant_rollouts():
    repo = configure_repo()
    for environment in ENVIRONMENT:
        model_name = MODEL_BY_ENVIRONMENT[environment][0]
        model, _ = torch.hub.load(
            str(repo),
            model_name,
            source="local",
            pretrained=True,
            device="cuda:0",
            trust_repo=True,
        )
        model.eval()
        predictor = set_action_path_trainability(model, False)
        predictor.use_activation_checkpointing = False
        base_state = extract_action_path_state(predictor)
        initial_cache = cache_initial_encodings(
            model, environment, list(range(NUM_STATES))
        )
        # Deliberately differs from the projection that carried adaptation
        # gradients. This blocks projection-specific codebook shortcuts.
        projector = CountSketchProjector(
            16 * 16 * int(predictor.predictor_embed_dim),
            READOUT_PROJECTION_DIM,
            EVALUATION_PROJECTION_SEED
            + 17 * ENVIRONMENT.index(environment),
        )
        with np.load(GOAL_ROOT / f"{model_name}.npz") as goals:
            goal_visual = goals["visual"].astype(np.float32)

        for variant in EVALUATION_VARIANTS:
            if variant == "frozen":
                load_action_path_state(predictor, base_state)
            else:
                checkpoint = torch.load(
                    ADAPTED_DIR / f"{model_name}_{variant}.pt",
                    map_location="cpu",
                    weights_only=False,
                )
                load_action_path_state(
                    predictor, checkpoint["action_path"]
                )
            for state_id in range(NUM_STATES):
                path = rollout_feature_path(
                    model_name, variant, state_id
                )
                path.parent.mkdir(parents=True, exist_ok=True)
                if path.exists():
                    continue
                with np.load(
                    TRANSITION_ROOT
                    / model_name
                    / f"state_{state_id:04d}.npz"
                ) as transition:
                    actions = transition["normalized_action"].astype(
                        np.float32
                    )
                    task_id = int(transition["task_id"])
                initial = initial_to_cuda(initial_cache[state_id])
                action_tensor = torch.as_tensor(
                    actions, device="cuda", dtype=torch.float32
                )
                with torch.inference_mode():
                    predicted, displacement, predicted_proprio = (
                        differentiable_unroll(
                            model, initial, action_tensor
                        )
                    )
                    batch, steps, tokens, channels = predicted.shape
                    projected = projector(
                        predicted.reshape(
                            batch * steps, tokens, channels
                        )
                    ).reshape(batch, steps, -1)
                    goal = torch.as_tensor(
                        goal_visual[task_id],
                        device="cuda",
                        dtype=torch.float32,
                    )
                    latent_cost = torch.sqrt(
                        torch.mean(
                            (
                                predicted
                                - goal[None, None, :, :]
                            )
                            ** 2,
                            dim=(2, 3),
                        ).clamp_min(1e-12)
                    )
                atomic_npz_uncompressed(
                    path,
                    projected=projected.detach().cpu().numpy().astype(
                        np.float16
                    ),
                    latent_cost=latent_cost.detach().cpu().numpy().astype(
                        np.float32
                    ),
                    predicted_proprio=predicted_proprio.detach()
                    .cpu()
                    .numpy()
                    .astype(np.float16),
                )
            log.info("%s cached variant %s", model_name, variant)
        del model, predictor, initial_cache
        gc.collect()
        torch.cuda.empty_cache()


def load_truth_evaluation(environment):
    payload = {
        "pose": [],
        "physical_cost": [],
        "interactions": [],
        "task_id": [],
        "split": [],
        "evaluation_seed": [],
    }
    for state_id in range(NUM_STATES):
        with np.load(
            TRUTH_ROOT
            / environment.lower()
            / f"state_{state_id:04d}.npz"
        ) as shard:
            payload["pose"].append(
                pose_target(environment, shard["all_endpoint_states"])[
                    :, HORIZON_INDICES
                ]
            )
            payload["physical_cost"].append(shard["physical_cost"])
            payload["interactions"].append(shard["interactions"])
            payload["task_id"].append(int(shard["task_id"]))
            payload["split"].append(str(shard["task_split"]))
            payload["evaluation_seed"].append(
                int(shard["evaluation_seed"])
            )
    return {
        key: np.asarray(value)
        for key, value in payload.items()
    }


def load_variant_features(model_name, variant):
    projected = []
    latent_cost = []
    for state_id in range(NUM_STATES):
        with np.load(
            rollout_feature_path(model_name, variant, state_id)
        ) as shard:
            projected.append(
                shard["projected"][:, HORIZON_INDICES].astype(np.float32)
            )
            latent_cost.append(
                shard["latent_cost"][:, HORIZON_INDICES].astype(
                    np.float64
                )
            )
    return np.asarray(projected), np.asarray(latent_cost)


def split_indices(truth, name):
    return np.flatnonzero(truth["split"] == name)


def flatten_selected(values, indices):
    selected = np.asarray(values)[indices]
    return selected.reshape(-1, selected.shape[-1])


def evaluate_stage9_methods():
    unit_rows = []
    probe_rows = []
    tasks_lookup = {
        environment: {
            task["task_id"]: task for task in TASKS[environment]
        }
        for environment in ENVIRONMENT
    }

    for environment in ENVIRONMENT:
        model_name = MODEL_BY_ENVIRONMENT[environment][0]
        truth = load_truth_evaluation(environment)
        train_ids = split_indices(truth, "probe_train")
        calibration_ids = split_indices(
            truth, "probe_calibration"
        )
        development_ids = split_indices(
            truth, DEVELOPMENT_SPLIT
        )
        method_costs = {}
        method_poses = {}

        for variant in EVALUATION_VARIANTS:
            features, latent_cost = load_variant_features(
                model_name, variant
            )
            probe = fit_linear_readout(
                flatten_selected(features, train_ids),
                flatten_selected(truth["pose"], train_ids),
                flatten_selected(features, calibration_ids),
                flatten_selected(truth["pose"], calibration_ids),
            )
            predicted_pose = predict_linear_readout(
                probe, features.reshape(-1, features.shape[-1])
            ).reshape(*features.shape[:-1], truth["pose"].shape[-1])
            method_poses[f"{variant}_linear_pose"] = predicted_pose
            method_costs[f"{variant}_latent_distance"] = latent_cost
            probe_rows.append(
                {
                    "environment": environment,
                    "model": model_name,
                    "variant": variant,
                    "ridge": probe["ridge"],
                    "calibration_pose_mse": probe[
                        "calibration_pose_mse"
                    ],
                    "readout_projection_dim": READOUT_PROJECTION_DIM,
                    "evaluation_projection_seed": (
                        EVALUATION_PROJECTION_SEED
                        + 17 * ENVIRONMENT.index(environment)
                    ),
                }
            )

        for state_id in development_ids:
            task_id = int(truth["task_id"][state_id])
            task = tasks_lookup[environment][task_id]
            true_cost = truth["physical_cost"][state_id]
            true_pose = truth["pose"][state_id]
            costs = {
                key: value[state_id]
                for key, value in method_costs.items()
            }
            for key, value in method_poses.items():
                costs[key] = decoded_task_cost(
                    environment, value[state_id], task
                )
            costs["oracle_pose"] = true_cost.copy()

            for horizon_index, horizon in enumerate(HORIZONS):
                for method, predicted in costs.items():
                    ranking = ranking_metrics(
                        true_cost[:, horizon_index],
                        predicted[:, horizon_index],
                    )
                    if method.endswith("_linear_pose"):
                        pose_prediction = method_poses[method][
                            state_id, :, horizon_index
                        ]
                        pose_error = float(
                            np.mean(
                                physical_pose_error(
                                    environment,
                                    pose_prediction,
                                    true_pose[:, horizon_index],
                                )
                            )
                        )
                    else:
                        pose_error = float("nan")
                    unit_rows.append(
                        {
                            "environment": environment,
                            "model": model_name,
                            "state_id": int(state_id),
                            "task_id": task_id,
                            "evaluation_seed": int(
                                truth["evaluation_seed"][state_id]
                            ),
                            "split": DEVELOPMENT_SPLIT,
                            "method": method,
                            "horizon": horizon,
                            "normalized_regret": ranking[
                                "normalized_regret"
                            ],
                            "weighted_pairwise_accuracy": ranking[
                                "weighted_pairwise_accuracy"
                            ],
                            "top1_correct": ranking["top1_correct"],
                            "normalized_margin_rmse": ranking[
                                "normalized_margin_rmse"
                            ],
                            "pose_error": pose_error,
                            "selected_action": ranking[
                                "selected_action"
                            ],
                            "oracle_action": ranking["oracle_action"],
                            "selected_is_null": int(
                                ranking["selected_action"] == 0
                            ),
                            "interaction_count_selected": int(
                                truth["interactions"][
                                    state_id,
                                    ranking["selected_action"],
                                    horizon_index,
                                ]
                            ),
                        }
                    )

    write_csv(OUT / "stage9_unit_metrics.csv", unit_rows)
    write_csv(OUT / "stage9_probe_selection.csv", probe_rows)
    return unit_rows


def summarize_and_contrast(unit_rows):
    summary = []
    fields = [
        "normalized_regret",
        "weighted_pairwise_accuracy",
        "top1_correct",
        "normalized_margin_rmse",
        "pose_error",
    ]
    keys = sorted(
        {
            (row["environment"], row["method"], int(row["horizon"]))
            for row in unit_rows
        }
    )
    for environment, method, horizon in keys:
        selected = [
            row
            for row in unit_rows
            if row["environment"] == environment
            and row["method"] == method
            and int(row["horizon"]) == horizon
        ]
        for metric in fields:
            result = bootstrap_mean(
                [float(row[metric]) for row in selected],
                [int(row["state_id"]) for row in selected],
                BOOTSTRAP_REPS,
                SEED
                + 101 * ENVIRONMENT.index(environment)
                + 13 * HORIZONS.index(horizon),
            )
            summary.append(
                {
                    "environment": environment,
                    "method": method,
                    "horizon": horizon,
                    "metric": metric,
                    **result,
                }
            )

    contrasts = []
    comparisons = [
        (
            "matched_cave_action_path_linear_pose",
            "frozen_linear_pose",
        ),
        (
            "matched_cave_action_path_linear_pose",
            "latent_only_action_path_linear_pose",
        ),
        (
            "matched_cave_action_path_linear_pose",
            "shuffled_cave_action_path_linear_pose",
        ),
        (
            "matched_cave_action_path_latent_distance",
            "frozen_latent_distance",
        ),
    ]
    for environment in ENVIRONMENT:
        for treatment, baseline in comparisons:
            for horizon in HORIZONS:
                for metric in [
                    "normalized_regret",
                    "weighted_pairwise_accuracy",
                    "top1_correct",
                ]:
                    treatment_rows = {
                        int(row["state_id"]): float(row[metric])
                        for row in unit_rows
                        if row["environment"] == environment
                        and row["method"] == treatment
                        and int(row["horizon"]) == horizon
                    }
                    baseline_rows = {
                        int(row["state_id"]): float(row[metric])
                        for row in unit_rows
                        if row["environment"] == environment
                        and row["method"] == baseline
                        and int(row["horizon"]) == horizon
                    }
                    common = sorted(
                        set(treatment_rows) & set(baseline_rows)
                    )
                    sign = -1.0 if metric == "normalized_regret" else 1.0
                    values = [
                        sign
                        * (
                            treatment_rows[state_id]
                            - baseline_rows[state_id]
                        )
                        for state_id in common
                    ]
                    result = bootstrap_mean(
                        values,
                        common,
                        BOOTSTRAP_REPS,
                        SEED + 7001 + 17 * HORIZONS.index(horizon),
                    )
                    contrasts.append(
                        {
                            "environment": environment,
                            "treatment": treatment,
                            "baseline": baseline,
                            "horizon": horizon,
                            "metric": metric,
                            "positive_means_treatment_better": True,
                            **result,
                        }
                    )
    write_csv(OUT / "stage9_metrics_summary.csv", summary)
    write_csv(OUT / "stage9_method_contrasts.csv", contrasts)
    return summary, contrasts


if not PIPELINE_FAILED:
    try:
        cache_variant_rollouts()
        STAGE9_UNIT_ROWS = evaluate_stage9_methods()
        STAGE9_SUMMARY, STAGE9_CONTRASTS = summarize_and_contrast(
            STAGE9_UNIT_ROWS
        )
    except Exception:
        record_failure("fresh_readout_and_development_evaluation")


In [ ]:
# Phase E — decision gate and compact diagnostics.


def contrast_lookup(contrasts, environment, treatment, baseline, metric):
    selected = [
        row
        for row in contrasts
        if row["environment"] == environment
        and row["treatment"] == treatment
        and row["baseline"] == baseline
        and row["metric"] == metric
    ]
    return selected


def stage9_decision():
    treatment = "matched_cave_action_path_linear_pose"
    frozen = "frozen_linear_pose"
    latent_only = "latent_only_action_path_linear_pose"
    shuffled = "shuffled_cave_action_path_linear_pose"
    gates = {}
    for environment in ENVIRONMENT:
        comparisons = {}
        for label, baseline in [
            ("versus_frozen", frozen),
            ("versus_latent_only", latent_only),
            ("versus_shuffled", shuffled),
        ]:
            regret = contrast_lookup(
                STAGE9_CONTRASTS,
                environment,
                treatment,
                baseline,
                "normalized_regret",
            )
            ranking = contrast_lookup(
                STAGE9_CONTRASTS,
                environment,
                treatment,
                baseline,
                "weighted_pairwise_accuracy",
            )
            comparisons[label] = {
                "regret_positive_horizons": int(
                    sum(float(row["estimate"]) > 0 for row in regret)
                ),
                "ranking_positive_horizons": int(
                    sum(float(row["estimate"]) > 0 for row in ranking)
                ),
                "regret_all": regret,
                "ranking_all": ranking,
            }
        gates[environment] = comparisons

    continue_gate = all(
        gates[environment]["versus_frozen"][
            "regret_positive_horizons"
        ]
        >= 2
        and gates[environment]["versus_frozen"][
            "ranking_positive_horizons"
        ]
        >= 2
        and gates[environment]["versus_latent_only"][
            "regret_positive_horizons"
        ]
        >= 2
        and gates[environment]["versus_shuffled"][
            "ranking_positive_horizons"
        ]
        >= 2
        for environment in ENVIRONMENT
    )
    payload = {
        "evidence_status": EVIDENCE_STATUS,
        "decision": (
            "ADVANCE_TO_NEW_TASK_CONFIRMATION"
            if continue_gate
            else "DO_NOT_CONFIRM_YET"
        ),
        "criterion": (
            "matched CAVE must improve regret and ranking at >=2 horizons "
            "in both environments, beat latent-only, and beat shuffled"
        ),
        "gates": gates,
        "interpretation_guardrail": (
            "This inspected task family is development evidence only. "
            "No positive result is confirmatory."
        ),
    }
    write_json(OUT / "stage9_development_decision.json", payload)
    return payload


def plot_stage9():
    methods = [
        "frozen_linear_pose",
        "latent_only_action_path_linear_pose",
        "shuffled_cave_action_path_linear_pose",
        "matched_cave_action_path_linear_pose",
    ]
    labels = ["Frozen", "Latent only", "Shuffled CAVE", "Matched CAVE"]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
    for row_index, environment in enumerate(ENVIRONMENT):
        for column_index, metric in enumerate(
            ["normalized_regret", "weighted_pairwise_accuracy"]
        ):
            axis = axes[row_index, column_index]
            for method, label in zip(methods, labels):
                values = []
                for horizon in HORIZONS:
                    selected = [
                        float(row["estimate"])
                        for row in STAGE9_SUMMARY
                        if row["environment"] == environment
                        and row["method"] == method
                        and int(row["horizon"]) == horizon
                        and row["metric"] == metric
                    ]
                    values.append(selected[0])
                axis.plot(HORIZONS, values, marker="o", label=label)
            axis.set_title(f"{environment}: {metric}")
            axis.set_xlabel("horizon")
            axis.grid(alpha=0.25)
    axes[0, 0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / "stage9_planning_comparison.png", dpi=180)
    plt.close(fig)


if not PIPELINE_FAILED:
    try:
        STAGE9_DECISION = stage9_decision()
        plot_stage9()
        print(json.dumps(STAGE9_DECISION, indent=2))
    except Exception:
        record_failure("stage9_decision_and_plots")


In [ ]:
# Phase F — package all non-cache outputs and download one result bundle.


def package_stage9_results():
    result_zip = Path("/content/stage9_result_bundle.zip")
    excluded_roots = {
        str(INTERMEDIATE.resolve()),
        str(CACHE_ROOT.resolve()),
    }
    files = []
    for path in OUT.rglob("*"):
        if not path.is_file():
            continue
        resolved = str(path.resolve())
        if any(
            resolved == root or resolved.startswith(root + os.sep)
            for root in excluded_roots
        ):
            continue
        files.append(path)
    manifest = [
        {
            "path": str(path.relative_to(OUT)),
            "size_bytes": int(path.stat().st_size),
            "sha256": sha256_file(path),
        }
        for path in sorted(files)
    ]
    write_json(
        OUT / "result_zip_manifest.json",
        {
            "run_signature": RUN_SIGNATURE,
            "pipeline_failed": bool(PIPELINE_FAILED),
            "files": manifest,
        },
    )
    files = sorted(
        {
            *files,
            OUT / "result_zip_manifest.json",
            OUT / "FAILURE_TRACE.txt",
        }
    )
    with zipfile.ZipFile(
        result_zip,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as archive:
        for path in files:
            if path.exists():
                archive.write(path, path.relative_to(OUT))
    print(f"RESULT_ZIP: {result_zip}")
    print("RUN_STATUS:", "FAILED" if PIPELINE_FAILED else "SUCCESS")
    if DOWNLOAD_RESULTS:
        from google.colab import files as colab_files

        colab_files.download(str(result_zip))
    return result_zip


try:
    RESULT_ZIP = package_stage9_results()
except Exception:
    record_failure("stage9_packaging")
    raise
